
# Explore ESPAC 2024 SQLite Database for Livestock

Livestock-focused notebook for building exploratory livestock, milk, and eggs LCI tables from `outputs/01_espac_2024.sqlite`.

This notebook is independent from the crop notebook and writes its own livestock summary and filtered CSV exports.


In [1]:

from pathlib import Path
import sqlite3
import json
import re
import unicodedata
from typing import List

import numpy as np
import pandas as pd
import ipywidgets as widgets
import yaml
from IPython.display import display, Markdown, HTML, clear_output

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'inputs').exists() and (PROJECT_DIR.parent / 'inputs').exists():
    PROJECT_DIR = PROJECT_DIR.parent
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
CSVS_DIR = OUTPUTS_DIR / 'CSVs'
CSVS_DIR.mkdir(parents=True, exist_ok=True)
DB_CANDIDATES = [CSVS_DIR / '01_espac_2024.sqlite', OUTPUTS_DIR / '01_espac_2024.sqlite']

def _count_tables(db_path: Path) -> int:
    if not db_path.exists():
        return -1
    with sqlite3.connect(db_path) as conn:
        return conn.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='table'").fetchone()[0]

existing_dbs = [p for p in DB_CANDIDATES if p.exists()]
assert existing_dbs, f'Database not found in any expected path: {DB_CANDIDATES}. Run the ETL notebook first.'
DB_PATH = max(existing_dbs, key=_count_tables)

# Guard against accidentally selecting an empty/corrupt placeholder database.
required_tables = {'rel_inec_glnac', 'rel_inec_gpnac', 'rel_inec_gvnac', 'rel_inec_oenac', 'rel_inec_apnac'}
with sqlite3.connect(DB_PATH) as conn:
    found_tables = {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()}
missing_tables = required_tables - found_tables
assert not missing_tables, (
    f'Invalid livestock database selected ({DB_PATH}). Missing required tables: {sorted(missing_tables)}'
)
print(f'Using DB: {DB_PATH}')

COEFF_CONFIG_PATH = PROJECT_DIR / 'inputs/02-05_espac_lci_coefficients.yml'
with COEFF_CONFIG_PATH.open('r', encoding='utf-8') as f:
    _cfg = yaml.safe_load(f) or {}

COSTA_PROVINCES = {
    "ESMERALDAS",
    "MANABI",
    "LOS RIOS",
    "GUAYAS",
    "SANTA ELENA",
    "EL ORO",
    "SANTO DOMINGO DE LOS TSACHILAS",
}
SIERRA_PROVINCES = {
    "AZUAY",
    "BOLIVAR",
    "CANAR",
    "CARCHI",
    "CHIMBORAZO",
    "COTOPAXI",
    "IMBABURA",
    "LOJA",
    "PICHINCHA",
    "TUNGURAHUA",
}
ORIENTE_PROVINCES = {
    "MORONA SANTIAGO",
    "NAPO",
    "ORELLANA",
    "PASTAZA",
    "SUCUMBIOS",
    "ZAMORA CHINCHIPE",
}

DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION = {
    "live_weight_kg_by_product_system": {
        "cattle_live": {"dairy": 450.0, "dual-purpose": 400.0, "beef": 400.0, "(unknown)": 400.0},
        "milk": {"dairy": 450.0, "dual-purpose": 400.0, "beef": 400.0, "(unknown)": 400.0},
        "swine_live": {"(all swine)": 75.23},
        "ovine_live": {"(all ovine)": 60.44},
        "goat_live": {"(all holdings)": 23.0},
        "eggs": {"layers": 1.514, "dual-purpose poultry": 1.514, "meat poultry": 1.514, "(unknown)": 1.514},
        "horse_live": {"(all holdings)": 312.0},
        "donkey_live": {"(all holdings)": 180.0},
        "mule_live": {"(all holdings)": 350.0},
    },
    "milk": {
        "normalization_product": "kg FPCM",
        "raw_milk_density_kg_per_l": 1.03,
        "default_fat_pct": 3.69,
        "default_true_protein_pct": 3.20,
        "kg_fpcm_per_l_raw_milk": 0.98273742,
        "liters_per_kg_fpcm": 1.017565025,
        "producing_years_ec": 5.5,
    },
    "eggs": {
        "normalization_product": "kg shell eggs",
        "egg_weight_kg_per_egg": 0.056699,
        "eggs_per_kg": 17.63699536,
        "output_periods_per_year": 52,
        "producing_years_ec": 1.5,
    },
}
LIVESTOCK_OUTPUT_NORMALIZATION = _cfg.get("livestock_output_normalization", DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION) or DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION
LIVE_WEIGHT_KG_BY_PRODUCT_SYSTEM = LIVESTOCK_OUTPUT_NORMALIZATION.get("live_weight_kg_by_product_system", DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["live_weight_kg_by_product_system"]) or {}
MEAT_YIELD_FRACTION_BY_PRODUCT_SYSTEM = LIVESTOCK_OUTPUT_NORMALIZATION.get("meat_yield_fraction_by_product_system", {}) or {}
_milk_norm_cfg = LIVESTOCK_OUTPUT_NORMALIZATION.get("milk", DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["milk"]) or DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["milk"]
MILK_KG_FPCM_PER_L = float(_milk_norm_cfg.get("kg_fpcm_per_l_raw_milk", DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["milk"]["kg_fpcm_per_l_raw_milk"]))
MILK_L_PER_KG_FPCM = float(_milk_norm_cfg.get("liters_per_kg_fpcm", DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["milk"]["liters_per_kg_fpcm"]))
MILK_PRODUCING_YEARS_EC = float(_milk_norm_cfg.get("producing_years_ec", DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["milk"].get("producing_years_ec", 1.0)))
if not np.isfinite(MILK_PRODUCING_YEARS_EC) or MILK_PRODUCING_YEARS_EC <= 0:
    MILK_PRODUCING_YEARS_EC = 1.0
_eggs_norm_cfg = LIVESTOCK_OUTPUT_NORMALIZATION.get("eggs", DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["eggs"]) or DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["eggs"]
EGG_WEIGHT_KG_PER_EGG = float(_eggs_norm_cfg.get("egg_weight_kg_per_egg", DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["eggs"]["egg_weight_kg_per_egg"]))
EGG_OUTPUT_PERIODS_PER_YEAR = float(_eggs_norm_cfg.get("output_periods_per_year", DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["eggs"].get("output_periods_per_year", 1.0)))
if not np.isfinite(EGG_OUTPUT_PERIODS_PER_YEAR) or EGG_OUTPUT_PERIODS_PER_YEAR <= 0:
    EGG_OUTPUT_PERIODS_PER_YEAR = 1.0
EGG_PRODUCING_YEARS_EC = float(_eggs_norm_cfg.get("producing_years_ec", DEFAULT_LIVESTOCK_OUTPUT_NORMALIZATION["eggs"].get("producing_years_ec", 1.0)))
if not np.isfinite(EGG_PRODUCING_YEARS_EC) or EGG_PRODUCING_YEARS_EC <= 0:
    EGG_PRODUCING_YEARS_EC = 1.0


def summary_strategy_token(summary_level: str) -> str:
    token = re.sub(r"[^a-z0-9]+", "_", str(summary_level).strip().lower()).strip("_")
    return token or 'province'


def format_numeric_for_display(df: pd.DataFrame, numeric_cols) -> pd.DataFrame:
    out = df.copy()
    for c in numeric_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce').round(6)
    return out


def display_scrollable_table(df: pd.DataFrame, max_height: str = '420px', index: bool = False):
    html = df.to_html(index=index, escape=False)
    wrapped = f"<div style='max-height:{max_height}; overflow:auto; border:1px solid #ddd'>{html}</div>"
    display(HTML(wrapped))


Using DB: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\01_espac_2024.sqlite


## 5) Livestock LCI extraction

This section builds livestock LCIs from the dairy cattle (`rel_inec_glnac`), swine (`rel_inec_gpnac`), ovine (`rel_inec_gvnac`), other livestock (`rel_inec_oenac`), and poultry (`rel_inec_apnac`) survey tables. It exports combined livestock summaries plus separate milk and eggs tables at province, region, and national levels, together with min/max uncertainty bounds from the underlying grouped records.

Logic used in this section:
- `cattle_live` and `milk` come from the ESPAC cattle module. Reported pasture and supplement shares are converted to approximate `kg feed/head/day` with configurable intake assumptions from `inputs/02-05_espac_lci_coefficients.yml`.
- `swine_live` comes from the swine module. Reported common-feed, supplement, and waste shares are converted to approximate `kg feed/head/day` with the same config-driven intake logic.
- `ovine_live`, `goat_live`, `horse_live`, `donkey_live`, `mule_live`, and `eggs` remain in the combined livestock table even when ESPAC does not expose diet shares. For those cases, the exporter attaches literature-based proxy diet columns from `livestock_diet_proxy_table` in `inputs/02-05_espac_lci_coefficients.yml`.
- Direct ESPAC diet fields and literature proxy diet fields are kept separate so downstream work can distinguish observed survey data from inferred proxy diets.
- Milk outputs are treated as reported daily liters when available (`gl_litlecvacaje`, fallback `gl_k813` / `litros_orde_ados`). Egg outputs remain in reported survey units because the questionnaire fields do not expose a clean conversion here.
- The interactive filter block at the end of the section lets the user manually preview a filtered livestock summary and export matching filtered summary / uncertainty CSVs by clicking `Export filtered CSV`, similar to the crop workflow.


In [2]:
from scripts.pipeline_manifest import (
    new_run_id,
    make_snapshot_copy,
    build_manifest_record,
    append_manifest_record,
)

DEFAULT_LIVESTOCK_FEED_ASSUMPTIONS = {
    "enabled": True,
    "unit_basis": "as-fed kg/head/day",
    "note": "Assumption-based total feed intake used to convert ESPAC feed-share fields into approximate absolute feed intensities for exploratory livestock LCIs.",
    "cattle_total_feed_kg_head_day": {
        "dairy": 12.0,
        "dual-purpose": 10.0,
        "beef": 8.0,
        "(unknown)": 10.0,
    },
    "swine_total_feed_kg_head_day": {
        "(all swine)": 2.5,
    },
}

LIVESTOCK_FEED_ASSUMPTIONS = _cfg.get("livestock_feed_assumptions", DEFAULT_LIVESTOCK_FEED_ASSUMPTIONS) or DEFAULT_LIVESTOCK_FEED_ASSUMPTIONS
CATTLE_TOTAL_FEED_KG_HEAD_DAY = {
    str(k): float(v)
    for k, v in (LIVESTOCK_FEED_ASSUMPTIONS.get("cattle_total_feed_kg_head_day") or DEFAULT_LIVESTOCK_FEED_ASSUMPTIONS["cattle_total_feed_kg_head_day"]).items()
}
SWINE_TOTAL_FEED_KG_HEAD_DAY = {
    str(k): float(v)
    for k, v in (LIVESTOCK_FEED_ASSUMPTIONS.get("swine_total_feed_kg_head_day") or DEFAULT_LIVESTOCK_FEED_ASSUMPTIONS["swine_total_feed_kg_head_day"]).items()
}
LIVESTOCK_DIET_UNIT_BASIS = str(LIVESTOCK_FEED_ASSUMPTIONS.get("unit_basis", DEFAULT_LIVESTOCK_FEED_ASSUMPTIONS["unit_basis"]))
DEFAULT_LIVESTOCK_DIET_PROXY_TABLE = {}
LIVESTOCK_DIET_PROXY_TABLE = _cfg.get("livestock_diet_proxy_table", DEFAULT_LIVESTOCK_DIET_PROXY_TABLE) or DEFAULT_LIVESTOCK_DIET_PROXY_TABLE
DEFAULT_LIVESTOCK_INFRA_WATER_ELECTRICITY_PROXY_TABLE = {}
LIVESTOCK_INFRA_WATER_ELECTRICITY_PROXY_TABLE = _cfg.get("livestock_infra_water_electricity_proxy_table", DEFAULT_LIVESTOCK_INFRA_WATER_ELECTRICITY_PROXY_TABLE) or DEFAULT_LIVESTOCK_INFRA_WATER_ELECTRICITY_PROXY_TABLE
LIVESTOCK_SUMMARY_LEVEL_OPTIONS = [
    ("By province", "province"),
    ("By region (confounded provinces)", "region"),
    ("By product, national (all regions/provinces/systems combined)", "product"),
    ("By product + system, national (confounded regions and provinces)", "national"),
]
LATEST_LIVESTOCK_FILTERED_SUMMARY_META_PATH = PROJECT_DIR / "outputs/02_latest_livestock_filtered_export_summary.json"


def _normalize_geo_name(value: str) -> str:
    x = unicodedata.normalize("NFKD", str(value or "").upper())
    x = "".join(ch for ch in x if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", x).strip()


def livestock_region_from_province(province: str) -> str:
    p = _normalize_geo_name(province)
    if p in COSTA_PROVINCES:
        return "Costa"
    if p in SIERRA_PROVINCES:
        return "Sierra"
    if p in ORIENTE_PROVINCES:
        return "Oriente"
    if p == "GALAPAGOS":
        return "Insular"
    return "(unknown)"


def mode_non_null(s: pd.Series):
    s2 = s.dropna()
    if len(s2) == 0:
        return np.nan
    m = s2.mode()
    return m.iloc[0] if len(m) else s2.iloc[0]


def get_livestock_group_keys(summary_level: str) -> List[str]:
    mapping = {
        "province": ["Region", "Province", "Product", "System"],
        "region": ["Region", "Product"],
        "product": ["Product"],
        "national": ["Product", "System"],
    }
    return mapping.get(summary_level, mapping["province"])


def _append_livestock_summary_placeholders(df: pd.DataFrame, summary_level: str) -> pd.DataFrame:
    out = df.copy()
    if "Region" not in out.columns:
        out.insert(0, "Region", "(all regions confounded)")
    if "Province" not in out.columns:
        insert_at = out.columns.get_loc("Region") + 1 if "Region" in out.columns else 0
        out.insert(insert_at, "Province", "(all provinces confounded)")
    if summary_level == "product" and "System" not in out.columns:
        insert_at = out.columns.get_loc("Product") + 1 if "Product" in out.columns else len(out.columns)
        out.insert(insert_at, "System", "(unknown)")
    return out


def write_latest_livestock_filtered_summary_metadata(summary_level: str, summary_token: str, filtered_csv_path: Path, filtered_unc_path: Path, run_id: str = "") -> None:
    payload = {
        "summary_level": str(summary_level),
        "summary_token": str(summary_token),
        "filtered_csv": str(filtered_csv_path),
        "filtered_uncertainty_csv": str(filtered_unc_path),
        "updated_at_utc": pd.Timestamp.utcnow().isoformat(),
        "run_id": str(run_id or ""),
    }
    LATEST_LIVESTOCK_FILTERED_SUMMARY_META_PATH.parent.mkdir(parents=True, exist_ok=True)
    LATEST_LIVESTOCK_FILTERED_SUMMARY_META_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def _safe_write_csv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_csv(path, index=False, encoding="utf-8-sig")
        return path
    except PermissionError:
        stamp = pd.Timestamp.utcnow().strftime("%Y%m%dT%H%M%SZ")
        fallback = path.with_name(f"{path.stem}__locked_{stamp}{path.suffix}")
        df.to_csv(fallback, index=False, encoding="utf-8-sig")
        print(f"File lock detected for {path.name}; wrote fallback export to {fallback.name} instead.")
        return fallback


def _coerce_espac_numeric(series: pd.Series) -> pd.Series:
    if series is None:
        return pd.Series(dtype=float)
    s = series.astype(str).str.strip()
    s = s.replace({"": np.nan, "None": np.nan, "nan": np.nan, "NaN": np.nan})
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def build_cattle_pasture_linkage_table(conn: sqlite3.Connection) -> pd.DataFrame:
    cultivated = pd.read_sql_query(
        "SELECT identificador, total, cp_k409ha FROM rel_inec_pcnac",
        conn,
    )
    if cultivated.empty:
        cultivated_summary = pd.DataFrame(columns=["identificador", "Cultivated_pasture_area_ha", "Cultivated_pasture_records_n"])
    else:
        cultivated["Cultivated_pasture_area_ha"] = _coerce_espac_numeric(cultivated.get("total"))
        fallback_area = _coerce_espac_numeric(cultivated.get("cp_k409ha"))
        cultivated["Cultivated_pasture_area_ha"] = cultivated["Cultivated_pasture_area_ha"].fillna(fallback_area)
        cultivated_summary = (
            cultivated.groupby("identificador", as_index=False)
            .agg(
                Cultivated_pasture_area_ha=("Cultivated_pasture_area_ha", "sum"),
                Cultivated_pasture_records_n=("identificador", "size"),
            )
        )

    natural = pd.read_sql_query(
        "SELECT identificador, su_categoria, supertotal, su_k202ha FROM rel_inec_sunac",
        conn,
    )
    if natural.empty:
        natural_summary = pd.DataFrame(columns=["identificador", "Natural_pasture_area_ha", "Natural_pasture_records_n"])
    else:
        natural["su_categoria_norm"] = natural["su_categoria"].astype(str).str.strip().str.upper()
        natural = natural.loc[natural["su_categoria_norm"].eq("PASTO NATURAL")].copy()
        natural["Natural_pasture_area_ha"] = _coerce_espac_numeric(natural.get("supertotal"))
        natural_fallback = _coerce_espac_numeric(natural.get("su_k202ha"))
        natural["Natural_pasture_area_ha"] = natural["Natural_pasture_area_ha"].fillna(natural_fallback)
        natural_summary = (
            natural.groupby("identificador", as_index=False)
            .agg(
                Natural_pasture_area_ha=("Natural_pasture_area_ha", "sum"),
                Natural_pasture_records_n=("identificador", "size"),
            )
        )

    pasture = cultivated_summary.merge(natural_summary, on="identificador", how="outer")
    if pasture.empty:
        return pasture

    for col in [
        "Cultivated_pasture_area_ha",
        "Cultivated_pasture_records_n",
        "Natural_pasture_area_ha",
        "Natural_pasture_records_n",
    ]:
        if col in pasture.columns:
            pasture[col] = pd.to_numeric(pasture[col], errors="coerce")

    pasture["Cultivated_pasture_area_ha"] = pasture.get("Cultivated_pasture_area_ha", 0).fillna(0.0)
    pasture["Natural_pasture_area_ha"] = pasture.get("Natural_pasture_area_ha", 0).fillna(0.0)
    pasture["Linked_pasture_area_ha"] = pasture["Cultivated_pasture_area_ha"] + pasture["Natural_pasture_area_ha"]
    pasture["Cultivated_pasture_area_share_pct"] = 100.0 * pasture["Cultivated_pasture_area_ha"] / pasture["Linked_pasture_area_ha"].replace(0, np.nan)
    pasture["Natural_pasture_area_share_pct"] = 100.0 * pasture["Natural_pasture_area_ha"] / pasture["Linked_pasture_area_ha"].replace(0, np.nan)
    return pasture


def build_cattle_pasture_linkage_diagnostic(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    cattle = df.loc[df.get("Product", pd.Series("", index=df.index)).astype(str).eq("cattle_live")].copy()
    if cattle.empty:
        return pd.DataFrame()

    cultivated_kg = pd.to_numeric(cattle.get("Cultivated_pasture_feed_kg_head_day"), errors="coerce").fillna(0.0)
    natural_kg = pd.to_numeric(cattle.get("Natural_pasture_feed_kg_head_day"), errors="coerce").fillna(0.0)
    unmatched_kg = pd.to_numeric(cattle.get("Unmatched_pasture_feed_kg_head_day"), errors="coerce").fillna(0.0)
    total_pasture_kg = pd.to_numeric(cattle.get("Pasture_feed_kg_head_day"), errors="coerce").fillna(0.0)

    cultivated_area = pd.to_numeric(cattle.get("Cultivated_pasture_area_ha"), errors="coerce").fillna(0.0)
    natural_area = pd.to_numeric(cattle.get("Natural_pasture_area_ha"), errors="coerce").fillna(0.0)
    linked_area = pd.to_numeric(cattle.get("Linked_pasture_area_ha"), errors="coerce").fillna(0.0)

    total_pasture_sum = total_pasture_kg.sum()
    linked_holdings = linked_area.gt(0)
    rows = [
        {
            "Component": "Cultivated pasture (linked pcnac)",
            "Pasture_feed_kg_head_day__sum": cultivated_kg.sum(),
            "Share_of_total_cattle_pasture_feed_pct": 100.0 * cultivated_kg.sum() / total_pasture_sum if total_pasture_sum > 0 else np.nan,
            "Area_ha__sum": cultivated_area.sum(),
            "Holdings_n": int(cultivated_area.gt(0).sum()),
        },
        {
            "Component": "Natural pasture (linked sunac)",
            "Pasture_feed_kg_head_day__sum": natural_kg.sum(),
            "Share_of_total_cattle_pasture_feed_pct": 100.0 * natural_kg.sum() / total_pasture_sum if total_pasture_sum > 0 else np.nan,
            "Area_ha__sum": natural_area.sum(),
            "Holdings_n": int(natural_area.gt(0).sum()),
        },
        {
            "Component": "Unmatched pasture residual",
            "Pasture_feed_kg_head_day__sum": unmatched_kg.sum(),
            "Share_of_total_cattle_pasture_feed_pct": 100.0 * unmatched_kg.sum() / total_pasture_sum if total_pasture_sum > 0 else np.nan,
            "Area_ha__sum": np.nan,
            "Holdings_n": int(total_pasture_kg.gt(0).sum() - linked_holdings.sum()),
        },
        {
            "Component": "Total reported cattle pasture feed",
            "Pasture_feed_kg_head_day__sum": total_pasture_sum,
            "Share_of_total_cattle_pasture_feed_pct": 100.0 if total_pasture_sum > 0 else np.nan,
            "Area_ha__sum": linked_area.sum(),
            "Holdings_n": int(total_pasture_kg.gt(0).sum()),
        },
    ]
    diag = pd.DataFrame(rows)
    diag["Diagnostic_note"] = (
        "Shares are calculated on the sum of cattle Pasture_feed_kg_head_day after splitting gl_porc_pasto across linked cultivated pasture area (pcnac), linked natural pasture area (sunac), and unmatched residual pasture. "
        "This is still an intake-assumption-based feed diagnostic, not a pasture biomass production balance."
    )
    return diag


def _apply_cattle_pasture_linkage(df: pd.DataFrame, pasture_link_df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if pasture_link_df is not None and not pasture_link_df.empty and "identificador" in out.columns:
        out = out.merge(pasture_link_df, on="identificador", how="left")

    defaults = {
        "Cultivated_pasture_area_ha": np.nan,
        "Natural_pasture_area_ha": np.nan,
        "Linked_pasture_area_ha": np.nan,
        "Cultivated_pasture_records_n": np.nan,
        "Natural_pasture_records_n": np.nan,
        "Cultivated_pasture_area_share_pct": np.nan,
        "Natural_pasture_area_share_pct": np.nan,
        "Cultivated_pasture_area_ha_per_head": np.nan,
        "Natural_pasture_area_ha_per_head": np.nan,
        "Cultivated_pasture_feed_share_pct": np.nan,
        "Natural_pasture_feed_share_pct": np.nan,
        "Unmatched_pasture_feed_share_pct": np.nan,
        "Cultivated_pasture_feed_kg_head_day": np.nan,
        "Natural_pasture_feed_kg_head_day": np.nan,
        "Unmatched_pasture_feed_kg_head_day": np.nan,
        "Pasture_linkage_status": "",
        "Pasture_linkage_note": "",
    }
    for col, default in defaults.items():
        if col not in out.columns:
            out[col] = default

    pasture_share = pd.to_numeric(out.get("Pasture_share_pct"), errors="coerce")
    pasture_kg = pd.to_numeric(out.get("Pasture_feed_kg_head_day"), errors="coerce")
    linked_total = pd.to_numeric(out.get("Linked_pasture_area_ha"), errors="coerce")
    cultivated_area = pd.to_numeric(out.get("Cultivated_pasture_area_ha"), errors="coerce").fillna(0.0)
    natural_area = pd.to_numeric(out.get("Natural_pasture_area_ha"), errors="coerce").fillna(0.0)

    cultivated_fraction = cultivated_area / linked_total.replace(0, np.nan)
    natural_fraction = natural_area / linked_total.replace(0, np.nan)

    out["Cultivated_pasture_area_ha_per_head"] = cultivated_area / pd.to_numeric(out.get("Animals_total_head"), errors="coerce").replace(0, np.nan)
    out["Natural_pasture_area_ha_per_head"] = natural_area / pd.to_numeric(out.get("Animals_total_head"), errors="coerce").replace(0, np.nan)

    out["Cultivated_pasture_feed_share_pct"] = pasture_share * cultivated_fraction
    out["Natural_pasture_feed_share_pct"] = pasture_share * natural_fraction
    out["Unmatched_pasture_feed_share_pct"] = pasture_share - out["Cultivated_pasture_feed_share_pct"].fillna(0) - out["Natural_pasture_feed_share_pct"].fillna(0)
    out["Unmatched_pasture_feed_share_pct"] = out["Unmatched_pasture_feed_share_pct"].clip(lower=0)
    out["Pasture_share_pct"] = (
        out["Cultivated_pasture_feed_share_pct"].fillna(0)
        + out["Natural_pasture_feed_share_pct"].fillna(0)
        + out["Unmatched_pasture_feed_share_pct"].fillna(0)
    ).where(pasture_share.notna(), np.nan)

    out["Cultivated_pasture_feed_kg_head_day"] = pasture_kg * cultivated_fraction
    out["Natural_pasture_feed_kg_head_day"] = pasture_kg * natural_fraction
    out["Unmatched_pasture_feed_kg_head_day"] = pasture_kg - out["Cultivated_pasture_feed_kg_head_day"].fillna(0) - out["Natural_pasture_feed_kg_head_day"].fillna(0)
    out["Unmatched_pasture_feed_kg_head_day"] = out["Unmatched_pasture_feed_kg_head_day"].clip(lower=0)
    out["Pasture_feed_kg_head_day"] = (
        out["Cultivated_pasture_feed_kg_head_day"].fillna(0)
        + out["Natural_pasture_feed_kg_head_day"].fillna(0)
        + out["Unmatched_pasture_feed_kg_head_day"].fillna(0)
    ).where(pasture_kg.notna(), np.nan)
    out["Common_feed_kg_head_day"] = pd.to_numeric(out.get("Common_feed_kg_head_day"), errors="coerce").fillna(0.0)
    out["Waste_feed_kg_head_day"] = pd.to_numeric(out.get("Waste_feed_kg_head_day"), errors="coerce").fillna(0.0)
    out["Total_feed_kg_head_day"] = (
        out["Pasture_feed_kg_head_day"].fillna(0)
        + pd.to_numeric(out.get("Supplement_feed_kg_head_day"), errors="coerce").fillna(0)
        + out["Common_feed_kg_head_day"].fillna(0)
        + out["Waste_feed_kg_head_day"].fillna(0)
        + pd.to_numeric(out.get("Unallocated_feed_kg_head_day"), errors="coerce").fillna(0)
    ).where(
        pasture_kg.notna()
        | pd.to_numeric(out.get("Supplement_feed_kg_head_day"), errors="coerce").notna()
        | pd.to_numeric(out.get("Unallocated_feed_kg_head_day"), errors="coerce").notna(),
        np.nan,
    )

    has_pasture = pasture_share.fillna(0) > 0
    has_cultivated = cultivated_area > 0
    has_natural = natural_area > 0
    has_any_link = has_cultivated | has_natural

    out["Pasture_linkage_status"] = np.select(
        [has_pasture & has_cultivated & has_natural, has_pasture & has_cultivated, has_pasture & has_natural, has_pasture & ~has_any_link],
        [
            "linked ESPAC cultivated and natural pasture areas",
            "linked ESPAC cultivated pasture area; residual pasture remains unmatched proxy",
            "linked ESPAC natural pasture area only; cultivated pasture not observed in pcnac",
            "no linked ESPAC pasture modules; pasture share remains unmatched proxy",
        ],
        default="no pasture share reported",
    )
    out["Pasture_linkage_note"] = (
        "Pasture share from gl_porc_pasto is split using Identificador-linked pasture area: pcnac total area for cultivated pasture and sunac PASTO NATURAL area for natural pasture. "
        "Any pasture share not matched to linked pasture modules remains in the unmatched pasture proxy component because ESPAC does not provide pasture biomass yield."
    )
    return out


def _apply_feed_share_to_kg_per_head_day(
    df: pd.DataFrame,
    intake_col: str,
    share_to_output_cols: List[tuple[str, str]],
    missing_status: str,
) -> pd.DataFrame:
    total = pd.to_numeric(df.get(intake_col), errors="coerce") if intake_col in df.columns else pd.Series(np.nan, index=df.index)
    known_sum = pd.Series(0.0, index=df.index)
    output_cols = []

    for share_col, output_col in share_to_output_cols:
        share = pd.to_numeric(df.get(share_col), errors="coerce") if share_col in df.columns else pd.Series(np.nan, index=df.index)
        df[output_col] = total * share / 100.0
        known_sum = known_sum.add(share.fillna(0), fill_value=0)
        output_cols.append(output_col)

    df["Unallocated_feed_share_pct"] = (100.0 - known_sum).where(known_sum > 0)
    df["Unallocated_feed_share_pct"] = df["Unallocated_feed_share_pct"].clip(lower=0)
    df["Unallocated_feed_kg_head_day"] = total * df["Unallocated_feed_share_pct"] / 100.0
    has_shares = known_sum > 0
    component_sum = pd.Series(0.0, index=df.index)
    for col in output_cols:
        component_sum = component_sum.add(pd.to_numeric(df.get(col), errors="coerce").fillna(0.0), fill_value=0)
    component_sum = component_sum.add(pd.to_numeric(df.get("Unallocated_feed_kg_head_day"), errors="coerce").fillna(0.0), fill_value=0)
    df["Total_feed_kg_head_day"] = component_sum.where(has_shares)
    df["Diet_data_status"] = np.where(
        has_shares,
        "share-based, absolute kg/head/day estimated from configured intake assumption",
        missing_status,
    )
    return df


def _init_missing_diet_columns(df: pd.DataFrame) -> pd.DataFrame:
    for c in [
        "Pasture_share_pct",
        "Cultivated_pasture_area_ha",
        "Natural_pasture_area_ha",
        "Linked_pasture_area_ha",
        "Cultivated_pasture_area_share_pct",
        "Natural_pasture_area_share_pct",
        "Cultivated_pasture_area_ha_per_head",
        "Natural_pasture_area_ha_per_head",
        "Cultivated_pasture_feed_share_pct",
        "Natural_pasture_feed_share_pct",
        "Unmatched_pasture_feed_share_pct",
        "Supplement_share_pct",
        "Common_feed_share_pct",
        "Waste_feed_share_pct",
        "Intake_assumption_kg_head_day",
        "Pasture_feed_kg_head_day",
        "Cultivated_pasture_feed_kg_head_day",
        "Natural_pasture_feed_kg_head_day",
        "Unmatched_pasture_feed_kg_head_day",
        "Supplement_feed_kg_head_day",
        "Common_feed_kg_head_day",
        "Waste_feed_kg_head_day",
        "Unallocated_feed_share_pct",
        "Unallocated_feed_kg_head_day",
        "Total_feed_kg_head_day",
    ]:
        if c not in df.columns:
            df[c] = np.nan
    return df


def _proxy_citation_string(citations) -> str:
    parts = []
    for item in citations or []:
        if not isinstance(item, dict):
            continue
        title = str(item.get("title", "")).strip()
        url = str(item.get("url", "")).strip()
        citation_note = str(item.get("citation_note", "")).strip()
        chunk = " - ".join([x for x in [title, citation_note] if x])
        if url:
            chunk = f"{chunk} ({url})" if chunk else url
        if chunk:
            parts.append(chunk)
    return " | ".join(parts)


def _proxy_component_value(proxy: dict, component: str, total_intake: float) -> float:
    components = proxy.get("proxy_components_kg_head_day") or {}
    if component in components and components.get(component) is not None:
        return float(components.get(component))

    shares = proxy.get("ration_split_pct") or {}
    share = shares.get(component)
    if share is None or pd.isna(share) or pd.isna(total_intake):
        return np.nan
    return float(total_intake) * float(share) / 100.0


def _map_proxy_diet_to_main_feed_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    product = out.get("Product", pd.Series("", index=out.index)).astype(str)
    proxy_total = pd.to_numeric(out.get("Proxy_total_feed_kg_head_day"), errors="coerce")
    current_total = pd.to_numeric(out.get("Total_feed_kg_head_day"), errors="coerce")
    proxy_mask = proxy_total.notna() & current_total.isna()
    if not proxy_mask.any():
        return out

    proxy_forage = pd.to_numeric(out.get("Proxy_forage_kg_head_day"), errors="coerce").fillna(0.0)
    proxy_tree = pd.to_numeric(out.get("Proxy_tree_forage_kg_head_day"), errors="coerce").fillna(0.0)
    proxy_compound = pd.to_numeric(out.get("Proxy_compound_feed_kg_head_day"), errors="coerce").fillna(0.0)
    proxy_mineral = pd.to_numeric(out.get("Proxy_mineral_kg_head_day"), errors="coerce").fillna(0.0)

    pasture_products = product.isin(["ovine_live", "goat_live"])
    poultry_products = product.isin(["eggs", "meat_poultry"])

    pasture_kg = np.where(pasture_products, proxy_forage + proxy_tree, 0.0)
    common_kg = np.where(poultry_products | pasture_products, proxy_compound, 0.0)
    supplement_kg = np.where(poultry_products | pasture_products, proxy_mineral, 0.0)
    waste_kg = np.zeros(len(out), dtype=float)
    unallocated_kg = np.clip(proxy_total.fillna(0.0) - pasture_kg - common_kg - supplement_kg - waste_kg, a_min=0.0, a_max=None)
    total_kg = pasture_kg + common_kg + supplement_kg + waste_kg + unallocated_kg

    proxy_total_safe = proxy_total.replace(0, np.nan)
    pasture_share = 100.0 * pd.Series(pasture_kg, index=out.index) / proxy_total_safe
    common_share = 100.0 * pd.Series(common_kg, index=out.index) / proxy_total_safe
    supplement_share = 100.0 * pd.Series(supplement_kg, index=out.index) / proxy_total_safe
    unallocated_share = 100.0 * pd.Series(unallocated_kg, index=out.index) / proxy_total_safe

    assignments = {
        "Intake_assumption_kg_head_day": proxy_total,
        "Pasture_feed_kg_head_day": pd.Series(pasture_kg, index=out.index),
        "Cultivated_pasture_feed_kg_head_day": pd.Series(np.zeros(len(out)), index=out.index),
        "Natural_pasture_feed_kg_head_day": pd.Series(np.zeros(len(out)), index=out.index),
        "Unmatched_pasture_feed_kg_head_day": pd.Series(pasture_kg, index=out.index),
        "Supplement_feed_kg_head_day": pd.Series(supplement_kg, index=out.index),
        "Common_feed_kg_head_day": pd.Series(common_kg, index=out.index),
        "Waste_feed_kg_head_day": pd.Series(waste_kg, index=out.index),
        "Unallocated_feed_kg_head_day": pd.Series(unallocated_kg, index=out.index),
        "Total_feed_kg_head_day": pd.Series(total_kg, index=out.index),
        "Pasture_share_pct": pasture_share,
        "Cultivated_pasture_feed_share_pct": pd.Series(np.zeros(len(out)), index=out.index),
        "Natural_pasture_feed_share_pct": pd.Series(np.zeros(len(out)), index=out.index),
        "Unmatched_pasture_feed_share_pct": pasture_share,
        "Supplement_share_pct": supplement_share,
        "Common_feed_share_pct": common_share,
        "Waste_feed_share_pct": pd.Series(np.zeros(len(out)), index=out.index),
        "Unallocated_feed_share_pct": unallocated_share.fillna(0.0),
    }
    for col, values in assignments.items():
        out.loc[proxy_mask, col] = pd.Series(values, index=out.index).loc[proxy_mask]

    if "Diet_basis" in out.columns and "Proxy_diet_basis" in out.columns:
        out.loc[proxy_mask & out["Diet_basis"].astype(str).str.strip().eq(""), "Diet_basis"] = out.loc[proxy_mask & out["Diet_basis"].astype(str).str.strip().eq(""), "Proxy_diet_basis"]
    if "Ration_composition_note" in out.columns:
        proxy_note = np.select(
            [poultry_products, pasture_products],
            [
                "main feed columns mapped from proxy diet as common compound feed, with no pasture or waste component",
                "main feed columns mapped from proxy diet as unmatched pasture for forage plus browse, with mineral or compound feed retained separately",
            ],
            default="main feed columns mapped from proxy diet components",
        )
        out.loc[proxy_mask, "Ration_composition_note"] = out.loc[proxy_mask, "Ration_composition_note"].astype(str).str.rstrip(".") + "; " + pd.Series(proxy_note, index=out.index).loc[proxy_mask]
    return out


def _apply_diet_proxy(df: pd.DataFrame, proxy_key: str) -> pd.DataFrame:
    proxy = LIVESTOCK_DIET_PROXY_TABLE.get(proxy_key) or {}
    proxy_cols = {
        "Proxy_total_feed_kg_head_day": np.nan,
        "Proxy_body_weight_kg": np.nan,
        "Proxy_forage_share_pct": np.nan,
        "Proxy_tree_forage_share_pct": np.nan,
        "Proxy_compound_feed_share_pct": np.nan,
        "Proxy_mineral_share_pct": np.nan,
        "Proxy_forage_kg_head_day": np.nan,
        "Proxy_tree_forage_kg_head_day": np.nan,
        "Proxy_compound_feed_kg_head_day": np.nan,
        "Proxy_mineral_kg_head_day": np.nan,
        "Proxy_diet_basis": "",
        "Diet_proxy_source_key": "",
        "Diet_proxy_quality": "",
        "Diet_proxy_note": "",
        "Diet_proxy_citations": "",
    }
    for col, default in proxy_cols.items():
        if col not in df.columns:
            df[col] = default

    if not proxy:
        return df

    total_intake = float(proxy.get("intake_kg_head_day")) if proxy.get("intake_kg_head_day") is not None else np.nan
    basis = str(proxy.get("intake_basis", "")).strip()
    shares = proxy.get("ration_split_pct") or {}

    df["Proxy_total_feed_kg_head_day"] = total_intake
    df["Proxy_body_weight_kg"] = pd.to_numeric(proxy.get("representative_body_weight_kg"), errors="coerce")
    df["Proxy_forage_share_pct"] = pd.to_numeric(shares.get("forage"), errors="coerce")
    df["Proxy_tree_forage_share_pct"] = pd.to_numeric(shares.get("tree_forage"), errors="coerce")
    df["Proxy_compound_feed_share_pct"] = pd.to_numeric(shares.get("compound_feed"), errors="coerce")
    df["Proxy_mineral_share_pct"] = pd.to_numeric(shares.get("mineral"), errors="coerce")
    df["Proxy_forage_kg_head_day"] = _proxy_component_value(proxy, "forage", total_intake)
    df["Proxy_tree_forage_kg_head_day"] = _proxy_component_value(proxy, "tree_forage", total_intake)
    df["Proxy_compound_feed_kg_head_day"] = _proxy_component_value(proxy, "compound_feed", total_intake)
    df["Proxy_mineral_kg_head_day"] = _proxy_component_value(proxy, "mineral", total_intake)
    df["Proxy_diet_basis"] = f"{basis} kg/head/day" if basis else ""
    df["Diet_proxy_source_key"] = str(proxy_key)
    df["Diet_proxy_quality"] = str(proxy.get("proxy_quality", "")).strip()
    df["Diet_proxy_note"] = str(proxy.get("note", "")).strip()
    df["Diet_proxy_citations"] = _proxy_citation_string(proxy.get("citations"))

    if "Diet_data_status" in df.columns:
        df["Diet_data_status"] = np.where(
            df["Diet_data_status"].astype(str).str.contains("no diet share data in ESPAC", case=False, na=False),
            "no diet share data in ESPAC; literature proxy attached from config",
            df["Diet_data_status"],
        )
    if "Ration_composition_note" in df.columns:
        df["Ration_composition_note"] = np.where(
            df["Ration_composition_note"].astype(str).str.strip().eq(""),
            "proxy diet/ration attached from inputs/02-05_espac_lci_coefficients.yml",
            df["Ration_composition_note"].astype(str).str.rstrip(".") + "; literature proxy diet attached from inputs/02-05_espac_lci_coefficients.yml",
        )
    return _map_proxy_diet_to_main_feed_columns(df)


def _resolve_live_weight_equivalent_kg(row: pd.Series) -> float:
    proxy_bw = pd.to_numeric(pd.Series([row.get("Proxy_body_weight_kg")]), errors="coerce").iloc[0]
    if pd.notna(proxy_bw) and proxy_bw > 0:
        return float(proxy_bw)
    product = str(row.get("Product", "")).strip()
    system = str(row.get("System", "")).strip()
    product_map = LIVE_WEIGHT_KG_BY_PRODUCT_SYSTEM.get(product, {}) or {}
    if system in product_map:
        return float(product_map[system])
    if "(unknown)" in product_map:
        return float(product_map["(unknown)"])
    return np.nan


def _resolve_meat_yield_fraction(row: pd.Series) -> float:
    product = str(row.get("Product", "")).strip()
    system = str(row.get("System", "")).strip()
    product_map = MEAT_YIELD_FRACTION_BY_PRODUCT_SYSTEM.get(product, {}) or {}
    if system in product_map:
        return float(product_map[system])
    if "(unknown)" in product_map:
        return float(product_map["(unknown)"])
    return np.nan


NORMALIZED_LIVESTOCK_EXPORT_SPECS = [
    ("Normalized_product_output_kg", None),
    ("Area_ha_per_1kg_product", "Area_ha"),
    ("Animals_total_live_weight_kg_per_1kg_product", "Animals_total_live_weight_kg"),
    ("Producing_animals_live_weight_kg_per_1kg_product", "Producing_animals_live_weight_kg"),
    ("Animals_sold_live_weight_kg_per_1kg_product", "Animals_sold_live_weight_kg"),
    ("Animal_input_age_related_factor", None),
    ("Animal_input_economic_allocation_factor", None),
    ("Animal_input_live_weight_kg_per_1kg_product", None),
    ("Animal_input_calf_live_weight_kg_per_1kg_product", None),
    ("Animal_input_one_day_chicken_live_weight_kg_per_1kg_product", None),
    ("Animal_input_piglet_live_weight_kg_per_1kg_product", None),
    ("Animal_input_kid_goat_live_weight_kg_per_1kg_product", None),
    ("Animal_input_dairy_cow_live_weight_kg_per_1kg_product", None),
    ("Animal_input_laying_hen_live_weight_kg_per_1kg_product", None),
    ("Water_l_per_1kg_product", "Water_l_head_day"),
    ("Electricity_kWh_per_1kg_product", "Electricity_kWh_head_day"),
    ("Live_weight_equivalent_kg_per_1kg_product", "Live_weight_equivalent_kg"),
    ("Milk_output_l_per_1kg_product", "Milk_output_l_day"),
    ("Milk_output_kg_fpcm_per_1kg_product", "Milk_output_kg_fpcm_day"),
    ("Milk_sales_l_per_1kg_product", "Milk_sales_l_7d"),
    ("Milk_sales_kg_fpcm_per_1kg_product", "Milk_sales_kg_fpcm_7d"),
    ("Egg_output_reported_per_1kg_product", "Egg_output_reported"),
    ("Egg_output_kg_per_1kg_product", "Egg_output_kg"),
    ("Egg_sales_reported_per_1kg_product", "Egg_sales_reported"),
    ("Egg_sales_kg_per_1kg_product", "Egg_sales_kg"),
    ("Birds_sold_head_per_1kg_product", "Birds_sold_head"),
    ("Birds_consumed_head_per_1kg_product", "Birds_consumed_head"),
    ("Cultivated_pasture_area_ha_per_1kg_product", "Cultivated_pasture_area_ha"),
    ("Natural_pasture_area_ha_per_1kg_product", "Natural_pasture_area_ha"),
    ("Linked_pasture_area_ha_per_1kg_product", "Linked_pasture_area_ha"),
    ("Intake_assumption_kg_per_1kg_product", "Intake_assumption_kg_head_day"),
    ("Pasture_feed_kg_per_1kg_product", "Pasture_feed_kg_head_day"),
    ("Cultivated_pasture_feed_kg_per_1kg_product", "Cultivated_pasture_feed_kg_head_day"),
    ("Natural_pasture_feed_kg_per_1kg_product", "Natural_pasture_feed_kg_head_day"),
    ("Unmatched_pasture_feed_kg_per_1kg_product", "Unmatched_pasture_feed_kg_head_day"),
    ("Supplement_feed_kg_per_1kg_product", "Supplement_feed_kg_head_day"),
    ("Common_feed_kg_per_1kg_product", "Common_feed_kg_head_day"),
    ("Waste_feed_kg_per_1kg_product", "Waste_feed_kg_head_day"),
    ("Unallocated_feed_kg_per_1kg_product", "Unallocated_feed_kg_head_day"),
    ("Total_feed_kg_per_1kg_product", "Total_feed_kg_head_day"),
    ("Proxy_total_feed_kg_per_1kg_product", "Proxy_total_feed_kg_head_day"),
    ("Proxy_forage_kg_per_1kg_product", "Proxy_forage_kg_head_day"),
    ("Proxy_tree_forage_kg_per_1kg_product", "Proxy_tree_forage_kg_head_day"),
    ("Proxy_compound_feed_kg_per_1kg_product", "Proxy_compound_feed_kg_head_day"),
    ("Proxy_mineral_kg_per_1kg_product", "Proxy_mineral_kg_head_day"),
    ("Vaccinated_head_per_1kg_product", "Vaccinated_head"),
]


def _append_normalized_livestock_inventory_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    norm_factor = pd.to_numeric(out.get("Normalization_factor_to_1kg"), errors="coerce")
    product = out.get("Product", pd.Series("", index=out.index)).astype(str)
    meat_yield_fraction = pd.to_numeric(out.get("Meat_yield_fraction"), errors="coerce")
    live_weight_products_without_meat_proxy = ~product.isin(["milk", "eggs"]) & meat_yield_fraction.isna()
    meat_products_with_proxy = ~product.isin(["milk", "eggs"]) & meat_yield_fraction.notna()
    out["Normalized_product_output_kg"] = np.where(norm_factor.notna(), 1.0, np.nan)
    for normalized_col, source_col in NORMALIZED_LIVESTOCK_EXPORT_SPECS:
        if source_col is None:
            continue
        if source_col in out.columns:
            out[normalized_col] = pd.to_numeric(out.get(source_col), errors="coerce") * norm_factor

    for col in [
        "Animals_total_live_weight_kg_per_1kg_product",
        "Producing_animals_live_weight_kg_per_1kg_product",
        "Animals_sold_live_weight_kg_per_1kg_product",
    ]:
        if col in out.columns:
            out.loc[live_weight_products_without_meat_proxy, col] = np.nan

    animal_input_columns = {
        "calf": "Animal_input_calf_live_weight_kg_per_1kg_product",
        "one_day_chicken": "Animal_input_one_day_chicken_live_weight_kg_per_1kg_product",
        "piglet": "Animal_input_piglet_live_weight_kg_per_1kg_product",
        "kid_goat": "Animal_input_kid_goat_live_weight_kg_per_1kg_product",
        "dairy_cow": "Animal_input_dairy_cow_live_weight_kg_per_1kg_product",
        "laying_hen": "Animal_input_laying_hen_live_weight_kg_per_1kg_product",
    }
    animal_input_economic_allocation_factors = {
        "calf": 0.0673,
        "one_day_chicken": 1.0,
        "piglet": 0.9372,
        "kid_goat": 1.0,
        "dairy_cow": 0.0376,
        "laying_hen": 1.0,
    }
    animal_input_key = np.select(
        [
            product.eq("milk"),
            product.eq("eggs"),
            product.eq("cattle_live"),
            product.eq("meat_poultry"),
            product.eq("swine_live"),
            product.isin(["goat_live", "ovine_live"]),
        ],
        ["dairy_cow", "laying_hen", "calf", "one_day_chicken", "piglet", "kid_goat"],
        default="",
    )
    animal_input_unallocated_value = pd.to_numeric(out.get("Animals_total_live_weight_kg_per_1kg_product"), errors="coerce")
    animal_input_age_factor = pd.to_numeric(out.get("Animal_input_age_related_factor"), errors="coerce")
    animal_input_age_factor = animal_input_age_factor.where(product.isin(["cattle_live", "swine_live", "ovine_live"]), 1.0)
    animal_input_unallocated_value = animal_input_unallocated_value * animal_input_age_factor
    animal_input_factor = pd.Series(animal_input_key, index=out.index).map(animal_input_economic_allocation_factors).astype(float)
    animal_input_value = animal_input_unallocated_value * animal_input_factor
    out["Animal_input_key"] = animal_input_key
    out["Animal_input_name"] = np.select(
        [animal_input_key == k for k in animal_input_columns],
        [
            "calf, at farm {BR} Economic, U",
            "one-day-chicken, at farm {BR} Economic, U",
            "piglet, at farm {BR} Economic, U",
            "kid goat, conventional, intensive forage area, at farm gate {FR} U",
            "dairy cow, at farm {BR} Economic, U",
            "laying hen <17 weeks, at farm {RER} Economic, U",
        ],
        default="",
    )
    out["Animal_input_economic_allocation_factor"] = animal_input_factor
    out["Animal_input_live_weight_kg_per_1kg_product"] = np.where(animal_input_key != "", animal_input_value, np.nan)
    for key, col in animal_input_columns.items():
        out[col] = np.where(animal_input_key == key, animal_input_value, 0.0)

    animal_input_feed_basis = pd.to_numeric(out.get("Animals_total_live_weight_kg_per_1kg_product"), errors="coerce")
    compound_feed_scale = (animal_input_value / animal_input_feed_basis).replace([np.inf, -np.inf], np.nan)
    compound_feed_scale = compound_feed_scale.where((animal_input_key != "") & compound_feed_scale.notna(), 1.0)
    for col in [
        "Supplement_feed_kg_per_1kg_product",
        "Common_feed_kg_per_1kg_product",
        "Waste_feed_kg_per_1kg_product",
        "Unallocated_feed_kg_per_1kg_product",
        "Proxy_compound_feed_kg_per_1kg_product",
    ]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce") * compound_feed_scale

    def _sum_existing_feed_components(columns: list[str]) -> pd.Series:
        existing = [col for col in columns if col in out.columns]
        if not existing:
            return pd.Series(np.nan, index=out.index)
        components = out[existing].apply(pd.to_numeric, errors="coerce")
        return components.sum(axis=1, min_count=1)

    out["Total_feed_kg_per_1kg_product"] = _sum_existing_feed_components([
        "Pasture_feed_kg_per_1kg_product",
        "Supplement_feed_kg_per_1kg_product",
        "Common_feed_kg_per_1kg_product",
        "Waste_feed_kg_per_1kg_product",
        "Unallocated_feed_kg_per_1kg_product",
    ])
    out["Proxy_total_feed_kg_per_1kg_product"] = _sum_existing_feed_components([
        "Proxy_forage_kg_per_1kg_product",
        "Proxy_tree_forage_kg_per_1kg_product",
        "Proxy_compound_feed_kg_per_1kg_product",
        "Proxy_mineral_kg_per_1kg_product",
    ])

    out["Animal_weight_normalization_note"] = np.select(
        [product.eq("milk"), product.eq("eggs"), meat_products_with_proxy],
        [
            "Animal kg per 1 kg product expresses the productive-life-amortized dairy-cow live-weight biomass associated with 1 kg FPCM, calculated from producing cows and the configured productive period.",
            "Animal kg per 1 kg product expresses the productive-life-amortized laying-hen live-weight biomass associated with 1 kg shell eggs, calculated from producing hens and the configured laying period.",
            "Animal live-weight kg per 1 kg product expresses the standing animal biomass associated with 1 kg meat output, using representative live weight and a carcass-yield proxy for the row.",
        ],
        default="For live-animal products without a meat-output proxy, animal-live-weight-kg-per-kg-product columns are left blank. Use Representative_live_weight_kg as the animal weight reference.",
    )
    out["Feed_mass_balance_note"] = (
        "Feed columns are constructed to satisfy: Pasture_feed_kg_per_1kg_product = Cultivated_pasture_feed_kg_per_1kg_product + Natural_pasture_feed_kg_per_1kg_product + Unmatched_pasture_feed_kg_per_1kg_product; and Total_feed_kg_per_1kg_product = Pasture_feed_kg_per_1kg_product + Supplement_feed_kg_per_1kg_product + Common_feed_kg_per_1kg_product + Waste_feed_kg_per_1kg_product + Unallocated_feed_kg_per_1kg_product. Non-pasture compound-feed components are scaled in proportion to the routed Animal_input_live_weight_kg_per_1kg_product before totals are recomputed."
    )
    return out


def _apply_product_normalization_basis(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    product = out.get("Product", pd.Series("", index=out.index)).astype(str)
    animals_total = pd.to_numeric(out.get("Animals_total_head"), errors="coerce") if "Animals_total_head" in out.columns else pd.Series(np.nan, index=out.index)
    producing = pd.to_numeric(out.get("Producing_animals_head"), errors="coerce") if "Producing_animals_head" in out.columns else pd.Series(np.nan, index=out.index)
    animals_sold = pd.to_numeric(out.get("Animals_sold_head"), errors="coerce") if "Animals_sold_head" in out.columns else pd.Series(np.nan, index=out.index)
    out["Representative_live_weight_kg"] = out.apply(_resolve_live_weight_equivalent_kg, axis=1)
    representative_live_weight = pd.to_numeric(out["Representative_live_weight_kg"], errors="coerce")
    out["Animals_total_live_weight_kg"] = animals_total * representative_live_weight
    out["Producing_animals_live_weight_kg"] = producing * representative_live_weight
    milk_mask = product.eq("milk")
    egg_mask = product.eq("eggs")
    out.loc[milk_mask, "Animals_total_live_weight_kg"] = out.loc[milk_mask, "Producing_animals_live_weight_kg"] / MILK_PRODUCING_YEARS_EC
    out.loc[egg_mask, "Animals_total_live_weight_kg"] = out.loc[egg_mask, "Producing_animals_live_weight_kg"] / EGG_PRODUCING_YEARS_EC
    feed_cols_to_amortize = [
        "Pasture_feed_kg_head_day",
        "Cultivated_pasture_feed_kg_head_day",
        "Natural_pasture_feed_kg_head_day",
        "Unmatched_pasture_feed_kg_head_day",
        "Supplement_feed_kg_head_day",
        "Common_feed_kg_head_day",
        "Waste_feed_kg_head_day",
        "Unallocated_feed_kg_head_day",
        "Total_feed_kg_head_day",
        "Proxy_total_feed_kg_head_day",
        "Proxy_forage_kg_head_day",
        "Proxy_tree_forage_kg_head_day",
        "Proxy_compound_feed_kg_head_day",
        "Proxy_mineral_kg_head_day",
    ]
    for feed_col in feed_cols_to_amortize:
        if feed_col in out.columns:
            out.loc[milk_mask, feed_col] = pd.to_numeric(out.loc[milk_mask, feed_col], errors="coerce") / MILK_PRODUCING_YEARS_EC
            out.loc[egg_mask, feed_col] = pd.to_numeric(out.loc[egg_mask, feed_col], errors="coerce") / EGG_PRODUCING_YEARS_EC
    out["Animals_sold_live_weight_kg"] = animals_sold * representative_live_weight
    meat_turnover_mask = product.isin(["cattle_live", "swine_live", "ovine_live"])
    animal_input_age_factor = animals_total / animals_sold.replace(0, np.nan)
    animal_input_age_factor = animal_input_age_factor.where((animals_total > 0) & (animals_sold > 0))
    out["Animal_input_age_related_factor"] = np.where(meat_turnover_mask, animal_input_age_factor, 1.0)
    out["Live_weight_equivalent_kg"] = out["Animals_total_live_weight_kg"]
    out["Meat_yield_fraction"] = out.apply(_resolve_meat_yield_fraction, axis=1)
    meat_yield_fraction = pd.to_numeric(out["Meat_yield_fraction"], errors="coerce")
    out["Estimated_meat_output_kg"] = out["Animals_total_live_weight_kg"] * meat_yield_fraction
    out["Milk_output_kg_fpcm_day"] = np.where(product.eq("milk"), pd.to_numeric(out.get("Milk_output_l_day"), errors="coerce") * MILK_KG_FPCM_PER_L, np.nan)
    out["Milk_sales_kg_fpcm_7d"] = np.where(product.eq("milk"), pd.to_numeric(out.get("Milk_sales_l_7d"), errors="coerce") * MILK_KG_FPCM_PER_L, np.nan)
    out["Egg_output_kg"] = np.where(product.eq("eggs"), pd.to_numeric(out.get("Egg_output_reported"), errors="coerce") * EGG_OUTPUT_PERIODS_PER_YEAR * EGG_WEIGHT_KG_PER_EGG, np.nan)
    out["Egg_sales_kg"] = np.where(product.eq("eggs"), pd.to_numeric(out.get("Egg_sales_reported"), errors="coerce") * EGG_OUTPUT_PERIODS_PER_YEAR * EGG_WEIGHT_KG_PER_EGG, np.nan)
    out["Reference_output_kg"] = np.select(
        [product.eq("milk"), product.eq("eggs"), meat_yield_fraction.notna()],
        [out["Milk_output_kg_fpcm_day"], out["Egg_output_kg"], out["Estimated_meat_output_kg"]],
        default=out["Live_weight_equivalent_kg"],
    )
    out["Reference_output_kg_for_dfe"] = out["Reference_output_kg"]
    out.loc[product.eq("milk"), "Reference_output_kg_for_dfe"] = out.loc[product.eq("milk"), "Milk_output_kg_fpcm_day"] * 365.0
    out["Normalization_factor_to_1kg"] = 1.0 / out["Reference_output_kg"].replace(0, np.nan)
    out["Functional_unit"] = np.select(
        [product.eq("milk"), product.eq("eggs"), meat_yield_fraction.notna()],
        ["1 kg FPCM", "1 kg shell eggs", "1 kg meat (carcass-weight proxy)"],
        default="1 kg live weight equivalent",
    )
    out["Reference_output_basis_note"] = np.select(
        [product.eq("milk"), product.eq("eggs"), meat_yield_fraction.notna()],
        [
            f"Milk liters/day converted to kg FPCM using {MILK_KG_FPCM_PER_L:.6f} kg FPCM/L; live-weight-equivalent per kg FPCM is based on total herd live weight divided by total milk output; DFE annualizes this daily output by multiplying by 365.",
            f"ESPAC egg counts are treated as weekly output and annualized with {EGG_OUTPUT_PERIODS_PER_YEAR:g} periods/year before conversion to kg shell eggs using {EGG_WEIGHT_KG_PER_EGG:.6f} kg/egg; laying-hen live weight and feed are amortized over {EGG_PRODUCING_YEARS_EC:.2f} producing years before normalization.",
            "Animals_total_head multiplied by representative live weight, then converted to estimated meat output with the configured carcass-yield proxy.",
        ],
        default="Animals_total_head multiplied by representative live weight to express kg live weight equivalent.",
    )
    out["Output_unit"] = np.select(
        [product.eq("milk"), product.eq("eggs"), meat_yield_fraction.notna()],
        ["kg FPCM/day", "kg shell eggs/year", "kg meat (carcass-weight proxy)"],
        default="kg live weight equivalent",
    )
    if "Sales_unit" in out.columns:
        out["Sales_unit"] = np.select(
            [product.eq("milk"), product.eq("eggs")],
            ["kg FPCM/7d", "kg shell eggs/year"],
            default=out["Sales_unit"],
        )
    out = _append_normalized_livestock_inventory_columns(out)
    return out


def _resolve_infra_water_electricity_proxy(product: str, system: str) -> dict:
    product_cfg = LIVESTOCK_INFRA_WATER_ELECTRICITY_PROXY_TABLE.get(str(product), {}) or {}
    if str(system) in product_cfg:
        return product_cfg.get(str(system), {}) or {}
    if "(unknown)" in product_cfg:
        return product_cfg.get("(unknown)", {}) or {}
    first_val = next(iter(product_cfg.values()), {}) if isinstance(product_cfg, dict) else {}
    return first_val or {}


def _apply_infra_water_electricity_proxy(df: pd.DataFrame) -> pd.DataFrame:
    defaults = {
        "Infrastructure_type": "",
        "Infrastructure_source": "",
        "Infrastructure_area_m2_head": np.nan,
        "Infrastructure_area_head_basis": "",
        "Infrastructure_area_ha": np.nan,
        "Water_l_head_day": np.nan,
        "Electricity_kWh_head_day": np.nan,
        "Infra_water_electricity_status": "",
        "Infra_water_electricity_note": "",
        "Infra_water_electricity_citations": "",
    }
    for col, default in defaults.items():
        if col not in df.columns:
            df[col] = default

    for idx, row in df.iterrows():
        proxy = _resolve_infra_water_electricity_proxy(row.get("Product", ""), row.get("System", ""))
        direct_infra = str(row.get("Milking_system_ESPAC", "")).strip()
        if direct_infra:
            df.at[idx, "Infrastructure_type"] = f"milking system: {direct_infra}"
            df.at[idx, "Infrastructure_source"] = "ESPAC direct (gl_sistema_ordenio)"
            df.at[idx, "Infra_water_electricity_status"] = "mixed: direct ESPAC infrastructure + literature proxy water/electricity"
            base_note = "infrastructure from ESPAC cattle milking-system field; water/electricity from literature proxy"
        else:
            df.at[idx, "Infrastructure_type"] = str(proxy.get("infrastructure_proxy", "")).strip()
            df.at[idx, "Infrastructure_source"] = "literature/system proxy"
            df.at[idx, "Infra_water_electricity_status"] = "literature/system proxy"
            base_note = "ESPAC livestock modules do not expose direct infrastructure/water/electricity variables for this row; literature/system proxy attached"

        df.at[idx, "Water_l_head_day"] = pd.to_numeric(proxy.get("water_l_head_day"), errors="coerce")
        df.at[idx, "Electricity_kWh_head_day"] = pd.to_numeric(proxy.get("electricity_kwh_head_day"), errors="coerce")
        area_m2_head = pd.to_numeric(proxy.get("area_m2_head"), errors="coerce")
        area_head_basis = str(proxy.get("area_head_basis", "")).strip()
        if pd.notna(area_m2_head) and area_m2_head > 0:
            basis_col = "Producing_animals_head" if area_head_basis == "producing_animals_head" else "Animals_total_head"
            basis_head = pd.to_numeric(pd.Series([row.get(basis_col)]), errors="coerce").iloc[0]
            area_ha = area_m2_head * basis_head / 10000.0 if pd.notna(basis_head) else np.nan
            df.at[idx, "Infrastructure_area_m2_head"] = area_m2_head
            df.at[idx, "Infrastructure_area_head_basis"] = area_head_basis or basis_col.lower()
            df.at[idx, "Infrastructure_area_ha"] = area_ha
            if "Area_ha" in df.columns and pd.isna(pd.to_numeric(pd.Series([df.at[idx, "Area_ha"]]), errors="coerce").iloc[0]):
                df.at[idx, "Area_ha"] = area_ha
        df.at[idx, "Infra_water_electricity_note"] = base_note
        df.at[idx, "Infra_water_electricity_citations"] = _proxy_citation_string(proxy.get("citations") or [])
    return df


def _livestock_export_columns(df: pd.DataFrame) -> List[str]:
    id_cols = [
        "Region",
        "Province",
        "Product",
        "Species",
        "System",
        "Functional_unit",
    ]
    normalized_cols = [c for c, _ in NORMALIZED_LIVESTOCK_EXPORT_SPECS]
    reference_cols = [
        "Area_ha",
        "Animals_total_head",
        "Producing_animals_head",
        "Animals_sold_head",
        "Animal_input_key",
        "Animal_input_name",
        "Milking_system_ESPAC",
        "Infrastructure_type",
        "Infrastructure_source",
        "Infrastructure_area_m2_head",
        "Infrastructure_area_head_basis",
        "Infrastructure_area_ha",
        "Water_l_head_day",
        "Electricity_kWh_head_day",
        "Representative_live_weight_kg",
        "Live_weight_equivalent_kg",
        "Milk_output_l_day",
        "Milk_output_kg_fpcm_day",
        "Milk_sales_l_7d",
        "Milk_sales_kg_fpcm_7d",
        "Milk_yield_l_head_day",
        "Egg_output_reported",
        "Egg_output_kg",
        "Egg_sales_reported",
        "Egg_sales_kg",
        "Eggs_per_head_reported",
        "Birds_sold_head",
        "Birds_consumed_head",
        "Pasture_share_pct",
        "Cultivated_pasture_area_ha",
        "Natural_pasture_area_ha",
        "Linked_pasture_area_ha",
        "Cultivated_pasture_area_share_pct",
        "Natural_pasture_area_share_pct",
        "Cultivated_pasture_area_ha_per_head",
        "Natural_pasture_area_ha_per_head",
        "Cultivated_pasture_feed_share_pct",
        "Natural_pasture_feed_share_pct",
        "Unmatched_pasture_feed_share_pct",
        "Supplement_share_pct",
        "Common_feed_share_pct",
        "Waste_feed_share_pct",
        "Intake_assumption_kg_head_day",
        "Pasture_feed_kg_head_day",
        "Cultivated_pasture_feed_kg_head_day",
        "Natural_pasture_feed_kg_head_day",
        "Unmatched_pasture_feed_kg_head_day",
        "Supplement_feed_kg_head_day",
        "Common_feed_kg_head_day",
        "Waste_feed_kg_head_day",
        "Unallocated_feed_share_pct",
        "Unallocated_feed_kg_head_day",
        "Total_feed_kg_head_day",
        "Proxy_total_feed_kg_head_day",
        "Proxy_body_weight_kg",
        "Proxy_forage_share_pct",
        "Proxy_tree_forage_share_pct",
        "Proxy_compound_feed_share_pct",
        "Proxy_mineral_share_pct",
        "Proxy_forage_kg_head_day",
        "Proxy_tree_forage_kg_head_day",
        "Proxy_compound_feed_kg_head_day",
        "Proxy_mineral_kg_head_day",
        "Vaccinated_head",
        "Vaccination_share_pct",
        "Infra_water_electricity_status",
        "Infra_water_electricity_note",
        "Infra_water_electricity_citations",
        "Reference_output_kg",
        "Reference_output_kg_for_dfe",
        "Normalization_factor_to_1kg",
        "Reference_output_basis_note",
        "Output_unit",
        "Sales_unit",
        "Diet_basis",
        "Proxy_diet_basis",
        "Diet_data_status",
        "Ration_composition_note",
        "Diet_proxy_source_key",
        "Diet_proxy_quality",
        "Diet_proxy_note",
        "Diet_proxy_citations",
    ]
    preferred = id_cols + normalized_cols + reference_cols
    return [c for c in preferred if c in df.columns]


def aggregate_livestock_lci(base_df: pd.DataFrame, summary_level: str = "province") -> pd.DataFrame:
    group_keys = get_livestock_group_keys(summary_level)
    work = base_df.copy()

    if "Region" not in group_keys:
        work["Region"] = "(all regions confounded)"
    if "Province" not in group_keys:
        work["Province"] = "(all provinces confounded)"

    numeric_cols = [
        c for c in work.columns
        if c not in group_keys and pd.api.types.is_numeric_dtype(work[c])
    ]
    text_cols = [
        c for c in [
            "Species",
            "Animal_input_key",
            "Animal_input_name",
            "Milking_system_ESPAC",
            "Infrastructure_type",
            "Infrastructure_source",
            "Functional_unit",
            "Infra_water_electricity_status",
            "Infra_water_electricity_note",
            "Infra_water_electricity_citations",
            "Pasture_linkage_status",
            "Pasture_linkage_note",
            "Reference_output_basis_note",
            "Output_unit",
            "Sales_unit",
            "Diet_basis",
            "Proxy_diet_basis",
            "Diet_data_status",
            "Ration_composition_note",
            "Diet_proxy_source_key",
            "Diet_proxy_quality",
            "Diet_proxy_note",
            "Diet_proxy_citations",
        ]
        if c in work.columns and c not in group_keys
    ]

    grouped = work.groupby(group_keys, as_index=False).size().rename(columns={"size": "count"})

    if numeric_cols:
        numeric_grouped = work.groupby(group_keys, as_index=False)[numeric_cols].median(numeric_only=True)
        grouped = grouped.merge(numeric_grouped, on=group_keys, how="left")

    if text_cols:
        text_grouped = work.groupby(group_keys, as_index=False)[text_cols].agg(mode_non_null)
        grouped = grouped.merge(text_grouped, on=group_keys, how="left")

    grouped = grouped.sort_values(group_keys).reset_index(drop=True).copy()
    grouped = _append_livestock_summary_placeholders(grouped, summary_level)
    return grouped[_livestock_export_columns(grouped)]


def build_livestock_uncertainty_table(base_df: pd.DataFrame, grouped_df: pd.DataFrame, summary_level: str = "province") -> pd.DataFrame:
    group_keys = get_livestock_group_keys(summary_level)
    work = base_df.copy()
    lower_q = 0.20
    upper_q = 0.80

    if "Region" not in group_keys:
        work["Region"] = "(all regions confounded)"
    if "Province" not in group_keys:
        work["Province"] = "(all provinces confounded)"

    numeric_cols = [
        c for c in grouped_df.columns
        if c not in group_keys and c != "count" and pd.api.types.is_numeric_dtype(grouped_df[c])
    ]
    # Some columns can be numeric in grouped_df but object/str in base_df due to mixed raw values.
    # Coerce before quantiles to avoid pandas TypeError on object dtypes.
    for c in numeric_cols:
        if c in work.columns:
            work[c] = pd.to_numeric(work[c], errors="coerce")

    if not numeric_cols:
        return _append_livestock_summary_placeholders(grouped_df[group_keys].copy(), summary_level)

    quantiles = work.groupby(group_keys)[numeric_cols].quantile([lower_q, upper_q]).unstack(level=-1)
    quantiles.columns = [
        f"{col}__minValue" if q == lower_q else f"{col}__maxValue"
        for col, q in quantiles.columns.to_flat_index()
    ]
    # De-fragment before reset_index to avoid pandas PerformanceWarning on fragmented frames.
    quantiles = quantiles.copy()
    unc = quantiles.reset_index()

    unc = unc.sort_values(group_keys).reset_index(drop=True)
    unc = _append_livestock_summary_placeholders(unc, summary_level)

    min_cols = [f"{c}__minValue" for c in numeric_cols if f"{c}__minValue" in unc.columns]
    max_cols = [f"{c}__maxValue" for c in numeric_cols if f"{c}__maxValue" in unc.columns]
    common_cols = [c for c in numeric_cols if c in grouped_df.columns and f"{c}__minValue" in unc.columns and f"{c}__maxValue" in unc.columns]
    if common_cols:
        mid_df = grouped_df[common_cols].apply(pd.to_numeric, errors="coerce")
        lo_df = unc[[f"{c}__minValue" for c in common_cols]].apply(pd.to_numeric, errors="coerce")
        hi_df = unc[[f"{c}__maxValue" for c in common_cols]].apply(pd.to_numeric, errors="coerce")
        lo_df.columns = common_cols
        hi_df.columns = common_cols

        min_clamped = np.minimum(np.minimum(lo_df, hi_df), mid_df)
        max_clamped = np.maximum(np.maximum(lo_df, hi_df), mid_df)

        min_clamped.columns = [f"{c}__minValue" for c in common_cols]
        max_clamped.columns = [f"{c}__maxValue" for c in common_cols]
        unc = pd.concat([unc.drop(columns=min_clamped.columns.tolist() + max_clamped.columns.tolist()), min_clamped, max_clamped], axis=1)

    live_min_col = "Live_weight_equivalent_kg__minValue"
    live_max_col = "Live_weight_equivalent_kg__maxValue"
    ref_min_col = "Reference_output_kg__minValue"
    ref_max_col = "Reference_output_kg__maxValue"
    input_min_col = "Animal_input_live_weight_kg_per_1kg_product__minValue"
    input_max_col = "Animal_input_live_weight_kg_per_1kg_product__maxValue"
    aggregate_bound_routed_inputs = {
        "goat_live": "Animal_input_kid_goat_live_weight_kg_per_1kg_product",
        "meat_poultry": "Animal_input_one_day_chicken_live_weight_kg_per_1kg_product",
    }
    product_unc = unc.get("Product", pd.Series("", index=unc.index)).astype(str)
    if all(c in unc.columns for c in [live_min_col, live_max_col, ref_min_col, ref_max_col]):
        aggregate_min = pd.to_numeric(unc[live_min_col], errors="coerce") / pd.to_numeric(unc[ref_max_col], errors="coerce").replace(0, np.nan)
        aggregate_max = pd.to_numeric(unc[live_max_col], errors="coerce") / pd.to_numeric(unc[ref_min_col], errors="coerce").replace(0, np.nan)
        for product_key, routed_base_col in aggregate_bound_routed_inputs.items():
            routed_min_col = f"{routed_base_col}__minValue"
            routed_max_col = f"{routed_base_col}__maxValue"
            routed_mask = product_unc.eq(product_key)
            if routed_mask.any() and routed_min_col in unc.columns and routed_max_col in unc.columns:
                unc.loc[routed_mask, routed_min_col] = aggregate_min.loc[routed_mask]
                unc.loc[routed_mask, routed_max_col] = aggregate_max.loc[routed_mask]
                if input_min_col in unc.columns and input_max_col in unc.columns:
                    unc.loc[routed_mask, input_min_col] = aggregate_min.loc[routed_mask]
                    unc.loc[routed_mask, input_max_col] = aggregate_max.loc[routed_mask]

    return unc


def build_cattle_live_lci_base(conn: sqlite3.Connection) -> pd.DataFrame:
    sql = """
        SELECT
            identificador,
            ual_prov,
            gl_supcrianza_ha,
            gl_k807,
            gl_k808,
            gl_porc_pasto,
            gl_porc_sobrealimento,
            gl_numvacunado,
            gl_sistema_ordenio,
            gl_propleche,
            gl_propdoblep,
            gl_propcarne
        FROM rel_inec_glnac
    """
    df = pd.read_sql_query(sql, conn)
    cattle_outflow = pd.read_sql_query(
        "SELECT identificador, total AS cattle_outflow_head FROM inec_vgvnac",
        conn,
    )
    if not cattle_outflow.empty:
        cattle_outflow["identificador"] = cattle_outflow["identificador"].astype(str).str.strip()
        cattle_outflow["cattle_outflow_head"] = _coerce_espac_numeric(cattle_outflow["cattle_outflow_head"])
        df["identificador"] = df["identificador"].astype(str).str.strip()
        df = df.merge(cattle_outflow, on="identificador", how="left")
    pasture_link_df = build_cattle_pasture_linkage_table(conn)
    df["Province"] = df["ual_prov"].astype(str).str.strip()
    df["Region"] = df["Province"].map(livestock_region_from_province)
    df["Product"] = "cattle_live"
    df["Species"] = "cattle"

    milk_flag = pd.to_numeric(df["gl_propleche"], errors="coerce").fillna(0)
    dual_flag = pd.to_numeric(df["gl_propdoblep"], errors="coerce").fillna(0)
    beef_flag = pd.to_numeric(df["gl_propcarne"], errors="coerce").fillna(0)
    df["System"] = np.select(
        [milk_flag > 0, dual_flag > 0, beef_flag > 0],
        ["dairy", "dual-purpose", "beef"],
        default="(unknown)",
    )

    df["Area_ha"] = pd.to_numeric(df["gl_supcrianza_ha"], errors="coerce")
    df["Animals_total_head"] = pd.to_numeric(df["gl_k808"], errors="coerce")
    df["Producing_animals_head"] = pd.to_numeric(df["gl_k807"], errors="coerce")
    df["Animals_sold_head"] = pd.to_numeric(df.get("cattle_outflow_head"), errors="coerce")
    df["Milking_system_ESPAC"] = df["gl_sistema_ordenio"].astype(str).str.strip()
    df.loc[df["Milking_system_ESPAC"].isin(["", "nan", "None"]), "Milking_system_ESPAC"] = ""
    df["Output_unit"] = "live head"
    df["Sales_unit"] = ""
    df["Diet_basis"] = LIVESTOCK_DIET_UNIT_BASIS

    df["Pasture_share_pct"] = pd.to_numeric(df["gl_porc_pasto"], errors="coerce")
    df["Supplement_share_pct"] = pd.to_numeric(df["gl_porc_sobrealimento"], errors="coerce")
    df["Common_feed_share_pct"] = np.nan
    df["Waste_feed_share_pct"] = np.nan
    df["Intake_assumption_kg_head_day"] = df["System"].map(lambda s: CATTLE_TOTAL_FEED_KG_HEAD_DAY.get(str(s), CATTLE_TOTAL_FEED_KG_HEAD_DAY.get("(unknown)", np.nan)))
    df = _apply_feed_share_to_kg_per_head_day(
        df,
        "Intake_assumption_kg_head_day",
        [
            ("Pasture_share_pct", "Pasture_feed_kg_head_day"),
            ("Supplement_share_pct", "Supplement_feed_kg_head_day"),
        ],
        missing_status="no diet share data in ESPAC",
    )
    df["Common_feed_kg_head_day"] = 0.0
    df["Waste_feed_kg_head_day"] = 0.0
    df["Vaccinated_head"] = pd.to_numeric(df["gl_numvacunado"], errors="coerce")
    df["Vaccination_share_pct"] = 100 * df["Vaccinated_head"] / df["Animals_total_head"].replace(0, np.nan)
    df = _apply_cattle_pasture_linkage(df, pasture_link_df)
    df["Ration_composition_note"] = np.where(
        df[["Pasture_share_pct", "Supplement_share_pct"]].notna().any(axis=1),
        "pasture + supplement shares from ESPAC cattle module; pasture share split across linked cultivated pasture, linked natural pasture, and unmatched residual using holding-level ESPAC pasture areas",
        "no diet/ration fields found in ESPAC cattle module",
    )

    df = _apply_infra_water_electricity_proxy(df)
    df = _apply_product_normalization_basis(df)
    keep_mask = df["Animals_total_head"].fillna(0) > 0
    return df.loc[keep_mask, _livestock_export_columns(df)].copy()


def build_milk_lci_base(conn: sqlite3.Connection) -> pd.DataFrame:
    sql = """
        SELECT
            identificador,
            ual_prov,
            gl_supcrianza_ha,
            gl_k807,
            gl_k808,
            vacas_ordenadas,
            gl_litlecvacaje,
            gl_k813,
            litros_orde_ados,
            gl_litlecven,
            gl_porc_pasto,
            gl_porc_sobrealimento,
            gl_numvacunado,
            gl_sistema_ordenio,
            gl_propleche,
            gl_propdoblep,
            gl_propcarne
        FROM rel_inec_glnac
    """
    df = pd.read_sql_query(sql, conn)
    pasture_link_df = build_cattle_pasture_linkage_table(conn)
    df["Province"] = df["ual_prov"].astype(str).str.strip()
    df["Region"] = df["Province"].map(livestock_region_from_province)
    df["Product"] = "milk"
    df["Species"] = "cattle"

    milk_flag = pd.to_numeric(df["gl_propleche"], errors="coerce").fillna(0)
    dual_flag = pd.to_numeric(df["gl_propdoblep"], errors="coerce").fillna(0)
    beef_flag = pd.to_numeric(df["gl_propcarne"], errors="coerce").fillna(0)
    df["System"] = np.select(
        [milk_flag > 0, dual_flag > 0, beef_flag > 0],
        ["dairy", "dual-purpose", "beef"],
        default="(unknown)",
    )

    df["Area_ha"] = pd.to_numeric(df["gl_supcrianza_ha"], errors="coerce")
    df["Animals_total_head"] = pd.to_numeric(df["gl_k808"], errors="coerce")
    df["Producing_animals_head"] = pd.to_numeric(df["vacas_ordenadas"], errors="coerce")
    df["Producing_animals_head"] = df["Producing_animals_head"].fillna(pd.to_numeric(df["gl_k807"], errors="coerce"))
    df["Animals_sold_head"] = np.nan
    df["Milking_system_ESPAC"] = df["gl_sistema_ordenio"].astype(str).str.strip()
    df.loc[df["Milking_system_ESPAC"].isin(["", "nan", "None"]), "Milking_system_ESPAC"] = ""

    df["Milk_output_l_day"] = pd.to_numeric(df["gl_litlecvacaje"], errors="coerce")
    df["Milk_output_l_day"] = df["Milk_output_l_day"].fillna(pd.to_numeric(df["gl_k813"], errors="coerce"))
    df["Milk_output_l_day"] = df["Milk_output_l_day"].fillna(pd.to_numeric(df["litros_orde_ados"], errors="coerce"))
    df["Milk_sales_l_7d"] = pd.to_numeric(df["gl_litlecven"], errors="coerce")
    df["Milk_yield_l_head_day"] = df["Milk_output_l_day"] / df["Producing_animals_head"].replace(0, np.nan)

    df["Output_unit"] = "L/day"
    df["Sales_unit"] = "L/7d"
    df["Diet_basis"] = LIVESTOCK_DIET_UNIT_BASIS
    df["Pasture_share_pct"] = pd.to_numeric(df["gl_porc_pasto"], errors="coerce")
    df["Supplement_share_pct"] = pd.to_numeric(df["gl_porc_sobrealimento"], errors="coerce")
    df["Common_feed_share_pct"] = np.nan
    df["Waste_feed_share_pct"] = np.nan
    df["Intake_assumption_kg_head_day"] = df["System"].map(lambda s: CATTLE_TOTAL_FEED_KG_HEAD_DAY.get(str(s), CATTLE_TOTAL_FEED_KG_HEAD_DAY.get("(unknown)", np.nan)))
    df = _apply_feed_share_to_kg_per_head_day(
        df,
        "Intake_assumption_kg_head_day",
        [
            ("Pasture_share_pct", "Pasture_feed_kg_head_day"),
            ("Supplement_share_pct", "Supplement_feed_kg_head_day"),
        ],
        missing_status="no diet share data in ESPAC",
    )
    df["Common_feed_kg_head_day"] = 0.0
    df["Waste_feed_kg_head_day"] = 0.0
    df["Vaccinated_head"] = pd.to_numeric(df["gl_numvacunado"], errors="coerce")
    df["Vaccination_share_pct"] = 100 * df["Vaccinated_head"] / df["Animals_total_head"].replace(0, np.nan)
    df = _apply_cattle_pasture_linkage(df, pasture_link_df)
    df["Ration_composition_note"] = np.where(
        df[["Pasture_share_pct", "Supplement_share_pct"]].notna().any(axis=1),
        "pasture + supplement shares from ESPAC cattle module; pasture share split across linked cultivated pasture, linked natural pasture, and unmatched residual using holding-level ESPAC pasture areas",
        "no diet/ration fields found in ESPAC cattle module",
    )

    df = _apply_infra_water_electricity_proxy(df)
    df = _apply_product_normalization_basis(df)
    keep_mask = (
        df["Milk_output_l_day"].fillna(0) > 0
    ) | (
        df["Milk_sales_l_7d"].fillna(0) > 0
    ) | (
        df["Producing_animals_head"].fillna(0) > 0
    )
    return df.loc[keep_mask, _livestock_export_columns(df)].copy()


def build_swine_lci_base(conn: sqlite3.Connection) -> pd.DataFrame:
    sql = """
        SELECT
            identificador,
            ual_prov,
            gp_superficie_ha,
            gp_k901,
            gp_totanio_ntp,
            gp_k910,
            gp_porc_alimento,
            gp_porc_sobrealimentacion,
            gp_porc_desechos
        FROM rel_inec_gpnac
    """
    df = pd.read_sql_query(sql, conn)
    df["Province"] = df["ual_prov"].astype(str).str.strip()
    df["Region"] = df["Province"].map(livestock_region_from_province)
    df["Product"] = "swine_live"
    df["Species"] = "swine"
    df["System"] = "(all swine)"
    df["Area_ha"] = pd.to_numeric(df["gp_superficie_ha"], errors="coerce")
    df["Animals_total_head"] = pd.to_numeric(df["gp_k901"], errors="coerce")
    df["Animals_total_head"] = df["Animals_total_head"].fillna(pd.to_numeric(df["gp_totanio_ntp"], errors="coerce"))
    df["Producing_animals_head"] = np.nan
    df["Animals_sold_head"] = pd.to_numeric(df["gp_k910"], errors="coerce")
    df["Output_unit"] = "live head"
    df["Sales_unit"] = "head"
    df["Diet_basis"] = LIVESTOCK_DIET_UNIT_BASIS

    df["Pasture_share_pct"] = np.nan
    df["Supplement_share_pct"] = pd.to_numeric(df["gp_porc_sobrealimentacion"], errors="coerce")
    df["Common_feed_share_pct"] = pd.to_numeric(df["gp_porc_alimento"], errors="coerce")
    df["Waste_feed_share_pct"] = pd.to_numeric(df["gp_porc_desechos"], errors="coerce")
    df["Intake_assumption_kg_head_day"] = SWINE_TOTAL_FEED_KG_HEAD_DAY.get("(all swine)", np.nan)
    df = _apply_feed_share_to_kg_per_head_day(
        df,
        "Intake_assumption_kg_head_day",
        [
            ("Common_feed_share_pct", "Common_feed_kg_head_day"),
            ("Supplement_share_pct", "Supplement_feed_kg_head_day"),
            ("Waste_feed_share_pct", "Waste_feed_kg_head_day"),
        ],
        missing_status="no diet share data in ESPAC",
    )
    df["Pasture_feed_kg_head_day"] = 0.0
    df["Cultivated_pasture_feed_kg_head_day"] = 0.0
    df["Natural_pasture_feed_kg_head_day"] = 0.0
    df["Unmatched_pasture_feed_kg_head_day"] = 0.0
    df["Pasture_share_pct"] = 0.0
    df["Cultivated_pasture_feed_share_pct"] = 0.0
    df["Natural_pasture_feed_share_pct"] = 0.0
    df["Unmatched_pasture_feed_share_pct"] = 0.0
    df["Vaccinated_head"] = np.nan
    df["Vaccination_share_pct"] = np.nan
    df["Ration_composition_note"] = np.where(
        df[["Common_feed_share_pct", "Supplement_share_pct", "Waste_feed_share_pct"]].notna().any(axis=1),
        "common feed + supplement + waste shares from ESPAC swine module",
        "no diet/ration fields found in ESPAC swine module",
    )

    df = _apply_infra_water_electricity_proxy(df)
    df = _apply_product_normalization_basis(df)
    keep_mask = df["Animals_total_head"].fillna(0) > 0
    return df.loc[keep_mask, _livestock_export_columns(df)].copy()


def build_ovine_lci_base(conn: sqlite3.Connection) -> pd.DataFrame:
    sql = """
        SELECT
            identificador,
            ual_prov,
            gv_superficie_ha,
            gv_k1001,
            gv_totanio,
            gv_k1010
        FROM rel_inec_gvnac
    """
    df = pd.read_sql_query(sql, conn)
    ovine_outflow = pd.read_sql_query(
        "SELECT identificador, total AS ovine_outflow_head FROM inec_vgonac",
        conn,
    )
    if not ovine_outflow.empty:
        ovine_outflow["identificador"] = ovine_outflow["identificador"].astype(str).str.strip()
        ovine_outflow["ovine_outflow_head"] = _coerce_espac_numeric(ovine_outflow["ovine_outflow_head"])
        df["identificador"] = df["identificador"].astype(str).str.strip()
        df = df.merge(ovine_outflow, on="identificador", how="left")
    df["Province"] = df["ual_prov"].astype(str).str.strip()
    df["Region"] = df["Province"].map(livestock_region_from_province)
    df["Product"] = "ovine_live"
    df["Species"] = "ovine"
    df["System"] = "(all ovine)"
    df["Area_ha"] = pd.to_numeric(df["gv_superficie_ha"], errors="coerce")
    df["Animals_total_head"] = pd.to_numeric(df["gv_k1001"], errors="coerce")
    df["Animals_total_head"] = df["Animals_total_head"].fillna(pd.to_numeric(df["gv_totanio"], errors="coerce"))
    df["Producing_animals_head"] = np.nan
    df["Animals_sold_head"] = pd.to_numeric(df.get("ovine_outflow_head"), errors="coerce")
    df["Animals_sold_head"] = df["Animals_sold_head"].fillna(pd.to_numeric(df["gv_k1010"], errors="coerce"))
    df["Output_unit"] = "live head"
    df["Sales_unit"] = "head"
    df["Diet_basis"] = ""
    df = _init_missing_diet_columns(df)
    df["Vaccinated_head"] = np.nan
    df["Vaccination_share_pct"] = np.nan
    df["Diet_data_status"] = "no diet share data in ESPAC"
    df["Ration_composition_note"] = "no diet/ration fields found in ovine module"
    df = _apply_diet_proxy(df, "ovine_live")

    df = _apply_infra_water_electricity_proxy(df)
    df = _apply_product_normalization_basis(df)
    keep_mask = df["Animals_total_head"].fillna(0) > 0
    return df.loc[keep_mask, _livestock_export_columns(df)].copy()


def build_other_livestock_lci_base(conn: sqlite3.Connection) -> pd.DataFrame:
    sql = """
        SELECT
            ual_prov,
            oe_superficie_ha,
            oe_k1101,
            oe_k1102,
            oe_k1103,
            oe_k1104
        FROM rel_inec_oenac
    """
    df = pd.read_sql_query(sql, conn)
    df["Province"] = df["ual_prov"].astype(str).str.strip()
    df["Region"] = df["Province"].map(livestock_region_from_province)
    df["Area_ha"] = pd.to_numeric(df["oe_superficie_ha"], errors="coerce")

    species_map = [
        ("oe_k1101", "donkey_live", "asnal"),
        ("oe_k1102", "horse_live", "equine"),
        ("oe_k1103", "mule_live", "mule"),
        ("oe_k1104", "goat_live", "caprine"),
    ]

    out_parts = []
    for col, product, species in species_map:
        tmp = df[["Region", "Province", "Area_ha", col]].copy()
        tmp["Animals_total_head"] = pd.to_numeric(tmp[col], errors="coerce")
        tmp = tmp[tmp["Animals_total_head"].fillna(0) > 0].copy()
        tmp["Product"] = product
        tmp["Species"] = species
        tmp["System"] = "(all holdings)"
        tmp["Producing_animals_head"] = np.nan
        tmp["Animals_sold_head"] = np.nan
        tmp["Output_unit"] = "live head"
        tmp["Sales_unit"] = ""
        tmp["Diet_basis"] = ""
        tmp = _init_missing_diet_columns(tmp)
        tmp["Vaccinated_head"] = np.nan
        tmp["Vaccination_share_pct"] = np.nan
        tmp["Diet_data_status"] = "no diet share data in ESPAC"
        tmp["Ration_composition_note"] = "no diet/ration fields found in other livestock module"
        tmp = _apply_diet_proxy(tmp, product)
        tmp = _apply_infra_water_electricity_proxy(tmp)
        tmp = _apply_product_normalization_basis(tmp)
        out_parts.append(tmp)

    out = pd.concat(out_parts, ignore_index=True, sort=False) if out_parts else pd.DataFrame()
    return out[_livestock_export_columns(out)].copy() if not out.empty else out


def build_poultry_lci_bases(conn: sqlite3.Connection) -> tuple[pd.DataFrame, pd.DataFrame]:
    sql = """
        SELECT
            ual_prov,
            ap_k1221,
            ap_k1222,
            ap_ctponedoras,
            ap_ctpollitos,
            ap_k1238,
            ap_k1240,
            ap_k051231,
            ap_cons_gapo,
            ap_vengapo,
            ap_venpoypo,
            ap_autopoypo
        FROM rel_inec_apnac
    """
    df = pd.read_sql_query(sql, conn)
    df["Province"] = df["ual_prov"].astype(str).str.strip()
    df["Region"] = df["Province"].map(livestock_region_from_province)
    df["Species"] = "chicken"
    df["Area_ha"] = np.nan

    activity = df["ap_k1221"].astype(str).str.strip()
    eggs_mask = ~activity.str.contains("carne", case=False, na=False)

    eggs_df = df.loc[eggs_mask].copy()
    eggs_df["Product"] = "eggs"
    eggs_df["System"] = np.select(
        [
            activity.loc[eggs_mask].str.contains("huevos", case=False, na=False),
            activity.loc[eggs_mask].str.contains("doble", case=False, na=False),
        ],
        ["layers", "dual-purpose poultry"],
        default="(unknown)",
    )
    eggs_df["Animals_total_head"] = pd.to_numeric(eggs_df["ap_k1222"], errors="coerce")
    eggs_df["Animals_total_head"] = eggs_df["Animals_total_head"].fillna(pd.to_numeric(eggs_df["ap_ctponedoras"], errors="coerce"))
    eggs_df["Producing_animals_head"] = eggs_df["Animals_total_head"]
    eggs_df["Animals_sold_head"] = np.nan
    eggs_df["Egg_output_reported"] = pd.to_numeric(eggs_df["ap_k1238"], errors="coerce")
    eggs_df["Egg_sales_reported"] = pd.to_numeric(eggs_df["ap_k1240"], errors="coerce")
    eggs_df["Egg_sales_reported"] = eggs_df["Egg_sales_reported"].fillna(pd.to_numeric(eggs_df["ap_vengapo"], errors="coerce"))
    eggs_df["Eggs_per_head_reported"] = eggs_df["Egg_output_reported"] / eggs_df["Producing_animals_head"].replace(0, np.nan)
    eggs_df["Birds_sold_head"] = pd.to_numeric(eggs_df["ap_k051231"], errors="coerce")
    eggs_df["Birds_consumed_head"] = pd.to_numeric(eggs_df["ap_cons_gapo"], errors="coerce")
    eggs_df["Output_unit"] = "reported weekly egg count"
    eggs_df["Sales_unit"] = "reported weekly egg count"
    eggs_df["Diet_basis"] = ""
    eggs_df = _init_missing_diet_columns(eggs_df)
    eggs_df["Vaccinated_head"] = np.nan
    eggs_df["Vaccination_share_pct"] = np.nan
    eggs_df["Diet_data_status"] = "no diet share data in ESPAC"
    eggs_df["Ration_composition_note"] = "no diet/ration fields found in avian module"
    eggs_df = _apply_diet_proxy(eggs_df, "eggs")
    eggs_df = _apply_infra_water_electricity_proxy(eggs_df)
    eggs_df = _apply_product_normalization_basis(eggs_df)
    eggs_keep_mask = (
        eggs_df["Egg_output_reported"].fillna(0) > 0
    ) | (
        eggs_df["Egg_sales_reported"].fillna(0) > 0
    ) | (
        eggs_df["Producing_animals_head"].fillna(0) > 0
    )
    eggs_df = eggs_df.loc[eggs_keep_mask, _livestock_export_columns(eggs_df)].copy()

    meat_df = df.loc[~eggs_mask].copy()
    meat_df["Product"] = "meat_poultry"
    meat_df["System"] = "(all holdings)"
    meat_df["Animals_total_head"] = pd.to_numeric(meat_df["ap_k1222"], errors="coerce")
    meat_df["Animals_total_head"] = meat_df["Animals_total_head"].fillna(pd.to_numeric(meat_df["ap_ctpollitos"], errors="coerce"))
    meat_df["Producing_animals_head"] = np.nan
    meat_sales = pd.to_numeric(meat_df["ap_venpoypo"], errors="coerce")
    meat_self = pd.to_numeric(meat_df["ap_autopoypo"], errors="coerce")
    meat_df["Animals_sold_head"] = meat_sales.fillna(0) + meat_self.fillna(0)
    meat_df["Animals_sold_head"] = meat_df["Animals_sold_head"].where(meat_sales.notna() | meat_self.notna(), np.nan)
    meat_df["Egg_output_reported"] = np.nan
    meat_df["Egg_sales_reported"] = np.nan
    meat_df["Eggs_per_head_reported"] = np.nan
    meat_df["Birds_sold_head"] = meat_sales
    meat_df["Birds_consumed_head"] = meat_self
    meat_df["Output_unit"] = "live head"
    meat_df["Sales_unit"] = "head"
    meat_df["Diet_basis"] = ""
    meat_df = _init_missing_diet_columns(meat_df)
    meat_df["Vaccinated_head"] = np.nan
    meat_df["Vaccination_share_pct"] = np.nan
    meat_df["Diet_data_status"] = "no diet share data in ESPAC"
    meat_df["Ration_composition_note"] = "no diet/ration fields found in avian module"
    meat_df = _apply_diet_proxy(meat_df, "meat_poultry")
    meat_df = _apply_infra_water_electricity_proxy(meat_df)
    meat_df = _apply_product_normalization_basis(meat_df)
    meat_keep_mask = (
        meat_df["Animals_total_head"].fillna(0) > 0
    ) | (
        meat_df["Animals_sold_head"].fillna(0) > 0
    )
    meat_df = meat_df.loc[meat_keep_mask, _livestock_export_columns(meat_df)].copy()

    return eggs_df, meat_df


with sqlite3.connect(DB_PATH) as conn:
    cattle_live_lci_base_df = build_cattle_live_lci_base(conn)
    milk_lci_base_df = build_milk_lci_base(conn)
    swine_lci_base_df = build_swine_lci_base(conn)
    ovine_lci_base_df = build_ovine_lci_base(conn)
    other_livestock_lci_base_df = build_other_livestock_lci_base(conn)
    eggs_lci_base_df, meat_poultry_lci_base_df = build_poultry_lci_bases(conn)

livestock_lci_base_df = pd.concat(
    [
        cattle_live_lci_base_df,
        milk_lci_base_df,
        swine_lci_base_df,
        ovine_lci_base_df,
        other_livestock_lci_base_df,
        eggs_lci_base_df,
        meat_poultry_lci_base_df,
    ],
    ignore_index=True,
    sort=False,
)
LIVESTOCK_SUMMARY_LEVELS = ["province", "region", "product", "national"]
livestock_exports = []

for _summary_level in LIVESTOCK_SUMMARY_LEVELS:
    _summary_token = summary_strategy_token(_summary_level)

    livestock_grouped_df = aggregate_livestock_lci(livestock_lci_base_df, summary_level=_summary_level)
    livestock_unc_df = build_livestock_uncertainty_table(livestock_lci_base_df, livestock_grouped_df, summary_level=_summary_level)

    milk_grouped_df = aggregate_livestock_lci(milk_lci_base_df, summary_level=_summary_level)
    milk_unc_df = build_livestock_uncertainty_table(milk_lci_base_df, milk_grouped_df, summary_level=_summary_level)

    eggs_grouped_df = aggregate_livestock_lci(eggs_lci_base_df, summary_level=_summary_level)
    eggs_unc_df = build_livestock_uncertainty_table(eggs_lci_base_df, eggs_grouped_df, summary_level=_summary_level)

    combined_path = CSVS_DIR / f"02_espac_livestock_lci_table__summary_{_summary_token}.csv"
    combined_unc_path = CSVS_DIR / f"02_espac_livestock_lci_table__summary_{_summary_token}_uncertainty.csv"
    milk_path = CSVS_DIR / f"02_espac_milk_lci_table__summary_{_summary_token}.csv"
    milk_unc_path = CSVS_DIR / f"02_espac_milk_lci_table__summary_{_summary_token}_uncertainty.csv"
    eggs_path = CSVS_DIR / f"02_espac_eggs_lci_table__summary_{_summary_token}.csv"
    eggs_unc_path = CSVS_DIR / f"02_espac_eggs_lci_table__summary_{_summary_token}_uncertainty.csv"

    combined_path_written = _safe_write_csv(livestock_grouped_df, combined_path)
    combined_unc_path_written = _safe_write_csv(livestock_unc_df, combined_unc_path)
    milk_path_written = _safe_write_csv(milk_grouped_df, milk_path)
    milk_unc_path_written = _safe_write_csv(milk_unc_df, milk_unc_path)
    eggs_path_written = _safe_write_csv(eggs_grouped_df, eggs_path)
    eggs_unc_path_written = _safe_write_csv(eggs_unc_df, eggs_unc_path)

    livestock_exports.append({
        "summary_level": _summary_level,
        "livestock_rows": len(livestock_grouped_df),
        "milk_rows": len(milk_grouped_df),
        "eggs_rows": len(eggs_grouped_df),
        "combined_path": str(combined_path_written),
        "combined_unc_path": str(combined_unc_path_written),
        "milk_path": str(milk_path_written),
        "milk_unc_path": str(milk_unc_path_written),
        "eggs_path": str(eggs_path_written),
        "eggs_unc_path": str(eggs_unc_path_written),
    })

_default_filtered_summary_level = "product"
_default_filtered_summary_token = summary_strategy_token(_default_filtered_summary_level)
_default_filtered_grouped_df = aggregate_livestock_lci(livestock_lci_base_df, summary_level=_default_filtered_summary_level)
_default_filtered_unc_df = build_livestock_uncertainty_table(
    livestock_lci_base_df,
    _default_filtered_grouped_df,
    summary_level=_default_filtered_summary_level,
)
_default_filtered_csv_path = CSVS_DIR / f"02_espac_livestock_lci_table_filtered__summary_{_default_filtered_summary_token}.csv"
_default_filtered_unc_path = CSVS_DIR / f"02_espac_livestock_lci_table_filtered__summary_{_default_filtered_summary_token}_uncertainty.csv"
_default_filtered_csv_path_written = _safe_write_csv(_default_filtered_grouped_df, _default_filtered_csv_path)
_default_filtered_unc_path_written = _safe_write_csv(_default_filtered_unc_df, _default_filtered_unc_path)
write_latest_livestock_filtered_summary_metadata(
    summary_level=_default_filtered_summary_level,
    summary_token=_default_filtered_summary_token,
    filtered_csv_path=_default_filtered_csv_path_written,
    filtered_unc_path=_default_filtered_unc_path_written,
)

print(f"Cattle live base rows: {len(cattle_live_lci_base_df):,}")
print(f"Milk base rows: {len(milk_lci_base_df):,}")
print(f"Swine base rows: {len(swine_lci_base_df):,}")
print(f"Ovine base rows: {len(ovine_lci_base_df):,}")
print(f"Other livestock base rows: {len(other_livestock_lci_base_df):,}")
print(f"Eggs base rows: {len(eggs_lci_base_df):,}")
print(f"Meat poultry base rows: {len(meat_poultry_lci_base_df):,}")
print(f"Combined livestock base rows: {len(livestock_lci_base_df):,}")
display(pd.DataFrame(livestock_exports))

display(Markdown("### Preview: combined livestock LCI (province summary)"))
province_preview = aggregate_livestock_lci(livestock_lci_base_df, summary_level="province")
display_scrollable_table(
    format_numeric_for_display(
        province_preview.head(25),
        [c for c in province_preview.columns if pd.api.types.is_numeric_dtype(province_preview[c])],
    ),
    max_height="420px",
)

display(Markdown("_Diet fields are populated in kg feed/head/day from ESPAC when feed shares are reported. For eggs, ovine, caprine, and equine-related holdings, the exporter now attaches literature-based proxy diet columns from `inputs/02-05_espac_lci_coefficients.yml` while keeping the direct ESPAC diet-share fields separate._"))

cattle_pasture_linkage_diag_df = build_cattle_pasture_linkage_diagnostic(livestock_lci_base_df)
CATTLE_PASTURE_LINKAGE_DIAG_CSV = CSVS_DIR / "02_espac_cattle_pasture_linkage_diagnostic.csv"
if not cattle_pasture_linkage_diag_df.empty:
    _safe_write_csv(cattle_pasture_linkage_diag_df, CATTLE_PASTURE_LINKAGE_DIAG_CSV)
    display(Markdown("### National cattle pasture linkage diagnostic"))
    display(
        format_numeric_for_display(
            cattle_pasture_linkage_diag_df,
            [c for c in cattle_pasture_linkage_diag_df.columns if pd.api.types.is_numeric_dtype(cattle_pasture_linkage_diag_df[c])],
        )
    )
    display(Markdown(f"Saved cattle pasture linkage diagnostic to `{CATTLE_PASTURE_LINKAGE_DIAG_CSV}`"))


livestock_region_filter = widgets.Dropdown(
    options=["All"] + sorted(livestock_lci_base_df["Region"].dropna().astype(str).unique().tolist()),
    value="All",
    description="Region:",
    layout=widgets.Layout(width="220px"),
)
livestock_summary_filter = widgets.Dropdown(
    options=LIVESTOCK_SUMMARY_LEVEL_OPTIONS,
    value="product",
    description="Summary:",
    layout=widgets.Layout(width="360px"),
)
livestock_province_filter = widgets.SelectMultiple(
    options=tuple(sorted(livestock_lci_base_df["Province"].dropna().astype(str).unique().tolist())),
    value=tuple(),
    description="Provinces:",
    rows=8,
    layout=widgets.Layout(width="320px"),
)
livestock_product_filter = widgets.SelectMultiple(
    options=tuple(sorted(livestock_lci_base_df["Product"].dropna().astype(str).unique().tolist())),
    value=tuple(),
    description="Products:",
    rows=8,
    layout=widgets.Layout(width="240px"),
)
livestock_species_filter = widgets.SelectMultiple(
    options=tuple(sorted(livestock_lci_base_df["Species"].dropna().astype(str).unique().tolist())),
    value=tuple(),
    description="Species:",
    rows=8,
    layout=widgets.Layout(width="220px"),
)
livestock_system_filter = widgets.SelectMultiple(
    options=tuple(sorted(livestock_lci_base_df["System"].dropna().astype(str).unique().tolist())),
    value=tuple(),
    description="Systems:",
    rows=8,
    layout=widgets.Layout(width="260px"),
)
livestock_rows_filter = widgets.IntSlider(value=30, min=10, max=300, step=10, description="Top rows:", continuous_update=False)
livestock_summary_out = widgets.Output()
livestock_filter_notes_out = widgets.Output()
livestock_export_out = widgets.Output()
livestock_export_btn = widgets.Button(description="Export filtered CSV", button_style="success", icon="download")

_livestock_callbacks_locked = False


def get_filtered_livestock_base_df() -> pd.DataFrame:
    df = livestock_lci_base_df.copy()
    if livestock_region_filter.value != "All":
        df = df[df["Region"] == livestock_region_filter.value]
    if livestock_province_filter.value:
        df = df[df["Province"].isin(list(livestock_province_filter.value))]
    if livestock_product_filter.value:
        df = df[df["Product"].isin(list(livestock_product_filter.value))]
    if livestock_species_filter.value:
        df = df[df["Species"].isin(list(livestock_species_filter.value))]
    if livestock_system_filter.value:
        df = df[df["System"].isin(list(livestock_system_filter.value))]
    return df


def get_filtered_livestock_grouped_df() -> pd.DataFrame:
    df = get_filtered_livestock_base_df()
    grouped = aggregate_livestock_lci(df, summary_level=livestock_summary_filter.value)
    sort_keys = [k for k in get_livestock_group_keys(livestock_summary_filter.value) if k in grouped.columns]
    return grouped.sort_values(sort_keys).reset_index(drop=True) if sort_keys else grouped.reset_index(drop=True)


def _sorted_unique(df: pd.DataFrame, col: str) -> tuple:
    if col not in df.columns:
        return tuple()
    return tuple(sorted(df[col].dropna().astype(str).unique().tolist()))


def refresh_livestock_selection_options(*_):
    global _livestock_callbacks_locked
    _livestock_callbacks_locked = True
    try:
        base = livestock_lci_base_df.copy()
        if livestock_region_filter.value != "All":
            base = base[base["Region"] == livestock_region_filter.value]

        provinces = _sorted_unique(base, "Province")
        current_provinces = tuple(v for v in livestock_province_filter.value if v in provinces)
        if tuple(livestock_province_filter.options) != provinces:
            livestock_province_filter.options = provinces
        if tuple(livestock_province_filter.value) != current_provinces:
            livestock_province_filter.value = current_provinces
        if livestock_province_filter.value:
            base = base[base["Province"].isin(list(livestock_province_filter.value))]

        products = _sorted_unique(base, "Product")
        current_products = tuple(v for v in livestock_product_filter.value if v in products)
        if tuple(livestock_product_filter.options) != products:
            livestock_product_filter.options = products
        if tuple(livestock_product_filter.value) != current_products:
            livestock_product_filter.value = current_products
        if livestock_product_filter.value:
            base = base[base["Product"].isin(list(livestock_product_filter.value))]

        species = _sorted_unique(base, "Species")
        current_species = tuple(v for v in livestock_species_filter.value if v in species)
        if tuple(livestock_species_filter.options) != species:
            livestock_species_filter.options = species
        if tuple(livestock_species_filter.value) != current_species:
            livestock_species_filter.value = current_species
        if livestock_species_filter.value:
            base = base[base["Species"].isin(list(livestock_species_filter.value))]

        systems = _sorted_unique(base, "System")
        current_systems = tuple(v for v in livestock_system_filter.value if v in systems)
        if tuple(livestock_system_filter.options) != systems:
            livestock_system_filter.options = systems
        if tuple(livestock_system_filter.value) != current_systems:
            livestock_system_filter.value = current_systems
    finally:
        _livestock_callbacks_locked = False


def refresh_livestock_outputs(*_):
    if _livestock_callbacks_locked:
        return

    livestock_summary_out.clear_output()
    livestock_filter_notes_out.clear_output()

    df = get_filtered_livestock_grouped_df()
    show_cols = [
        "count",
        "Region",
        "Province",
        "Product",
        "Species",
        "System",
        "Area_ha",
        "Animals_total_head",
        "Producing_animals_head",
        "Animals_sold_head",
        "Milk_output_l_day",
        "Milk_yield_l_head_day",
        "Egg_output_reported",
        "Pasture_share_pct",
        "Supplement_share_pct",
        "Common_feed_share_pct",
        "Waste_feed_share_pct",
        "Total_feed_kg_head_day",
        "Proxy_total_feed_kg_head_day",
        "Proxy_forage_share_pct",
        "Proxy_tree_forage_share_pct",
        "Proxy_compound_feed_share_pct",
        "Proxy_mineral_share_pct",
        "Proxy_forage_kg_head_day",
        "Proxy_tree_forage_kg_head_day",
        "Proxy_compound_feed_kg_head_day",
        "Proxy_mineral_kg_head_day",
        "Diet_basis",
        "Proxy_diet_basis",
        "Diet_data_status",
        "Diet_proxy_source_key",
        "Diet_proxy_quality",
    ]
    show_cols = [c for c in show_cols if c in df.columns]
    sort_keys = [k for k in get_livestock_group_keys(livestock_summary_filter.value) if k in df.columns]
    show = df[show_cols].sort_values(sort_keys).reset_index(drop=True) if sort_keys else df[show_cols].reset_index(drop=True)

    with livestock_summary_out:
        display(Markdown("### Livestock LCI table"))
        if show.empty:
            display(Markdown("No livestock rows after current filters."))
        else:
            display_scrollable_table(
                format_numeric_for_display(
                    show.head(livestock_rows_filter.value),
                    [c for c in show.columns if pd.api.types.is_numeric_dtype(show[c])],
                ),
                max_height="420px",
            )
            display(Markdown(f"Filtered grouped rows: **{len(show):,}**"))

    with livestock_filter_notes_out:
        filtered_base = get_filtered_livestock_base_df()
        display(Markdown("### Livestock logic notes"))
        display(Markdown(
            f"- Base rows after filters: **{len(filtered_base):,}**\n"
            f"- Summary level: **`{livestock_summary_filter.value}`**\n"
            f"- Direct ESPAC diet shares are used for cattle and swine when available.\n"
            f"- Literature proxy diet fields are attached for missing-species diets through `livestock_diet_proxy_table` in `inputs/02-05_espac_lci_coefficients.yml`.\n"
            f"- Filtered exports write grouped medians plus a matching min/max uncertainty file."
        ))


def on_livestock_selection_change(*_):
    if _livestock_callbacks_locked:
        return
    refresh_livestock_selection_options()
    refresh_livestock_outputs()


def export_filtered_livestock_csv(_):
    export_run_id = new_run_id("02_livestock")
    livestock_export_out.clear_output()
    filtered_grouped_df = get_filtered_livestock_grouped_df()
    filtered_base_df = get_filtered_livestock_base_df()
    summary_token = summary_strategy_token(livestock_summary_filter.value)

    out_path = CSVS_DIR / f"02_espac_livestock_lci_table_filtered__summary_{summary_token}.csv"
    filtered_grouped_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    filtered_unc_df = build_livestock_uncertainty_table(
        filtered_base_df,
        filtered_grouped_df,
        summary_level=livestock_summary_filter.value,
    )
    filtered_unc_path = CSVS_DIR / f"02_espac_livestock_lci_table_filtered__summary_{summary_token}_uncertainty.csv"
    filtered_unc_df.to_csv(filtered_unc_path, index=False, encoding="utf-8-sig")

    main_snapshot = make_snapshot_copy(out_path, export_run_id)
    unc_snapshot = make_snapshot_copy(filtered_unc_path, export_run_id)
    manifest_record = build_manifest_record(
        run_id=export_run_id,
        domain="livestock",
        summary_token=summary_token,
        pipeline_stage="02",
        source_main_csv=main_snapshot,
        source_unc_csv=unc_snapshot,
        main_df=filtered_grouped_df,
        unc_df=filtered_unc_df,
        filters_meta={
            "summary_level": livestock_summary_filter.value,
            "region": str(livestock_region_filter.value),
            "provinces": list(livestock_province_filter.value),
            "products": list(livestock_product_filter.value),
            "species": list(livestock_species_filter.value),
            "systems": list(livestock_system_filter.value),
        },
        extra={"otros_expansion_source": "none", "subcrop_count_by_label": {}},
    )
    manifest_path = append_manifest_record(PROJECT_DIR, manifest_record)

    write_latest_livestock_filtered_summary_metadata(
        summary_level=livestock_summary_filter.value,
        summary_token=summary_token,
        filtered_csv_path=out_path,
        filtered_unc_path=filtered_unc_path,
        run_id=export_run_id,
    )

    with livestock_export_out:
        display(Markdown(f"Saved **{len(filtered_grouped_df):,}** grouped livestock rows to `{out_path}`"))
        display(Markdown(f"Saved filtered livestock uncertainty bounds to `{filtered_unc_path}`"))
        display(Markdown(f"Updated livestock export metadata: `{LATEST_LIVESTOCK_FILTERED_SUMMARY_META_PATH}`"))
        display(Markdown(f"Manifest updated: `{manifest_path}` (`run_id={export_run_id}`)"))


for _w in [
    livestock_region_filter,
    livestock_summary_filter,
    livestock_province_filter,
    livestock_product_filter,
    livestock_species_filter,
    livestock_system_filter,
    livestock_rows_filter,
]:
    _w.observe(on_livestock_selection_change, names="value")
livestock_export_btn.on_click(export_filtered_livestock_csv)

refresh_livestock_selection_options()
refresh_livestock_outputs()

display(Markdown("### Filtered livestock export"))
display(widgets.HBox([livestock_region_filter, livestock_summary_filter]))
display(widgets.HBox([livestock_province_filter, livestock_product_filter, livestock_species_filter, livestock_system_filter]))
display(livestock_rows_filter)
display(livestock_export_btn)
display(livestock_summary_out)
display(livestock_filter_notes_out)
display(livestock_export_out)




Cattle live base rows: 12,441
Milk base rows: 10,168
Swine base rows: 4,599
Ovine base rows: 2,164
Other livestock base rows: 7,338
Eggs base rows: 251
Meat poultry base rows: 6
Combined livestock base rows: 36,967


C:\Users\AAVADI\AppData\Local\Temp\ipykernel_45240\2829289454.py:102: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "updated_at_utc": pd.Timestamp.utcnow().isoformat(),


,summary_level,livestock_rows,milk_rows,eggs_rows,combined_path,combined_unc_path,milk_path,milk_unc_path,eggs_path,eggs_unc_path
0,province,331,88,18,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...
1,region,35,4,3,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...
2,product,10,1,1,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...
3,national,18,4,3,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...,c:\Users\AAVADI\OneDrive - IRTA\Documents\_Pro...


### Preview: combined livestock LCI (province summary)

Region,Province,Product,Species,System,Functional_unit,Normalized_product_output_kg,Area_ha_per_1kg_product,Animals_total_live_weight_kg_per_1kg_product,Producing_animals_live_weight_kg_per_1kg_product,Animals_sold_live_weight_kg_per_1kg_product,Animal_input_age_related_factor,Animal_input_economic_allocation_factor,Animal_input_live_weight_kg_per_1kg_product,Animal_input_calf_live_weight_kg_per_1kg_product,Animal_input_one_day_chicken_live_weight_kg_per_1kg_product,Animal_input_piglet_live_weight_kg_per_1kg_product,Animal_input_kid_goat_live_weight_kg_per_1kg_product,Animal_input_dairy_cow_live_weight_kg_per_1kg_product,Animal_input_laying_hen_live_weight_kg_per_1kg_product,Water_l_per_1kg_product,Electricity_kWh_per_1kg_product,Live_weight_equivalent_kg_per_1kg_product,Milk_output_l_per_1kg_product,Milk_output_kg_fpcm_per_1kg_product,Milk_sales_l_per_1kg_product,Milk_sales_kg_fpcm_per_1kg_product,Egg_output_reported_per_1kg_product,Egg_output_kg_per_1kg_product,Egg_sales_reported_per_1kg_product,Egg_sales_kg_per_1kg_product,Birds_sold_head_per_1kg_product,Birds_consumed_head_per_1kg_product,Cultivated_pasture_area_ha_per_1kg_product,Natural_pasture_area_ha_per_1kg_product,Linked_pasture_area_ha_per_1kg_product,Intake_assumption_kg_per_1kg_product,Pasture_feed_kg_per_1kg_product,Cultivated_pasture_feed_kg_per_1kg_product,Natural_pasture_feed_kg_per_1kg_product,Unmatched_pasture_feed_kg_per_1kg_product,Supplement_feed_kg_per_1kg_product,Common_feed_kg_per_1kg_product,Waste_feed_kg_per_1kg_product,Unallocated_feed_kg_per_1kg_product,Total_feed_kg_per_1kg_product,Proxy_total_feed_kg_per_1kg_product,Proxy_forage_kg_per_1kg_product,Proxy_tree_forage_kg_per_1kg_product,Proxy_compound_feed_kg_per_1kg_product,Proxy_mineral_kg_per_1kg_product,Vaccinated_head_per_1kg_product,Area_ha,Animals_total_head,Producing_animals_head,Animals_sold_head,Animal_input_key,Animal_input_name,Milking_system_ESPAC,Infrastructure_type,Infrastructure_source,Infrastructure_area_m2_head,Infrastructure_area_ha,Water_l_head_day,Electricity_kWh_head_day,Representative_live_weight_kg,Live_weight_equivalent_kg,Milk_output_l_day,Milk_output_kg_fpcm_day,Milk_sales_l_7d,Milk_sales_kg_fpcm_7d,Milk_yield_l_head_day,Egg_output_reported,Egg_output_kg,Egg_sales_reported,Egg_sales_kg,Eggs_per_head_reported,Birds_sold_head,Birds_consumed_head,Pasture_share_pct,Cultivated_pasture_area_ha,Natural_pasture_area_ha,Linked_pasture_area_ha,Cultivated_pasture_area_share_pct,Natural_pasture_area_share_pct,Cultivated_pasture_area_ha_per_head,Natural_pasture_area_ha_per_head,Cultivated_pasture_feed_share_pct,Natural_pasture_feed_share_pct,Unmatched_pasture_feed_share_pct,Supplement_share_pct,Common_feed_share_pct,Waste_feed_share_pct,Intake_assumption_kg_head_day,Pasture_feed_kg_head_day,Cultivated_pasture_feed_kg_head_day,Natural_pasture_feed_kg_head_day,Unmatched_pasture_feed_kg_head_day,Supplement_feed_kg_head_day,Common_feed_kg_head_day,Waste_feed_kg_head_day,Unallocated_feed_share_pct,Unallocated_feed_kg_head_day,Total_feed_kg_head_day,Proxy_total_feed_kg_head_day,Proxy_body_weight_kg,Proxy_forage_share_pct,Proxy_tree_forage_share_pct,Proxy_compound_feed_share_pct,Proxy_mineral_share_pct,Proxy_forage_kg_head_day,Proxy_tree_forage_kg_head_day,Proxy_compound_feed_kg_head_day,Proxy_mineral_kg_head_day,Vaccinated_head,Vaccination_share_pct,Infra_water_electricity_status,Infra_water_electricity_note,Infra_water_electricity_citations,Reference_output_kg,Reference_output_kg_for_dfe,Normalization_factor_to_1kg,Reference_output_basis_note,Output_unit,Sales_unit,Diet_basis,Proxy_diet_basis,Diet_data_status,Ration_composition_note,Diet_proxy_source_key,Diet_proxy_quality,Diet_proxy_note,Diet_proxy_citations
(unknown),ZONA NO DELIMITADA,cattle_live,cattle,dairy,1 kg meat (carcass-weight proxy),1.0,0.004502,2.000000,0.769231,0.250000,8.000000,0.0673,1.076800,1.076800,0.0,0.000,0.000000,0.000000,0.000000,0.017794,0.000404,2.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002495,0.0,0.002495

...

IntSlider(value=30, continuous_update=False, description='Top rows:', max=300, min=10, step=10)

Button(button_style='success', description='Export filtered CSV', icon='download', style=ButtonStyle())

Output()

Output()

Output()